# Comparative Reliability Performance of N-Body Simulation under Hamiltonian Framework Applying Barnes-Hut Trees
---
Within the realms of modern orbital mechanics application of Barnes-Hut trees is a common practice for large N-Body simulations (Barnes & Hut, 1986). The project describes the creation of a C++ and R tool to simulate the problem assessing reliability of the Barnes-Hut trees for cosmological applications via application of symplectic and non-symplectic integration schemes.

## About The Notebook
This notebook is a simulation prototype, that shows fast results and validation of said results using Python.
It will be kept in a dated entry order expanding each individual feature for a later transfer to C++ and R framework.
It will also comply with the project's gantt chart of progress.

Please bear in mind that the function and variable names follow a certain pattern:
- Integrators and their daughter functions: `operation_integrator_order`
- Mathematical equations take their literal mathematical names, where numerator is a numerator, product is a product, etc.

Displayed equations are numbered consecutively through the whole notebook and are cited as Eq. (*n*). Sources are cited in the Author (Year) form and are listed in the References section at the end of the notebook.

# Symplectic Methods
---

## Setting the Environment and Creating Velocity Verlet 4th Order

In [ ]:
# Imports
import matplotlib.pyplot as plt
import numpy as np
import random as rnd
import time

# Randomisation Seed
seed = 256
rnd.seed(seed)

# Constants
# Yoshida composition constants
yoshida_x0 = -(2**(1/3))/(2-(2**(1/3)))
yoshida_x1 = (1/(2-(2**(1/3))))

In [ ]:
"""
Creating helper functions to help with force evaluation later.
"""

# Find Euclidian distance between two bodies
# obj_a – object i with accessible properties
# obj_b - object j with accessible properties
def euclidian_distance(obj_a, obj_b):
    return np.linalg.norm(obj_a.r - obj_b.r)

# Find initial velocity
# obj – individual object with accessible properties
# G – constant of gravitational attraction
# distance – previously found Euclidian distance
def vel_init(obj, G, distance):
    return rnd.random() * np.sqrt((G*obj.m) / distance)  # random variable in the beginning is used to create random orbit eccentricities (no ejection)

# Get a perpendicular vector
def get_perpendicular(v):
    x, y, z = v
    if x != 0 or y != 0:
        perp = np.array([-y, x, 0.0], dtype=float)
    else:
        perp = np.array([0.0, 1.0, 0.0], dtype=float)

    return perp

# Find velocity direction
# Essentially convert scalar velocity into vector velocity
# obj_a – object i with accessible properties
# obj_b - object j with accessible properties
# v – scalar velocity
def vel_direction(obj_a, obj_b, v):
    radius_vector = obj_b.r - obj_a.r
    perp_vector = get_perpendicular(radius_vector)
    unit_perp_vector = perp_vector / np.linalg.norm(perp_vector)    # normalise by its own length

    return v * unit_perp_vector


# Compute Gravitational Acceleration of All Bodies acting on i
# G – the constant of gravitational attraction
# bod_obj – a list of stellar bodies represented as objects.
#           Contains objects themselves. Allows to access
#           individual parameters such as: mass, initial velocity,
#           and initial displacement.
# e – softening factor to prevent division by 0 errors in case
#     the bodies get too close.
def acc(G, bod_obj, e):
    N = len(bod_obj)    
    accelerations = [np.zeros(3) for _ in range(N)]     # initialise an array to hold acceleration vectors for all bodies

    # Sigma operation
    # Loop over every body i
    for i in range(N):
        a_i = np.zeros(3)   # net acceleration vector on body i

        # Sum gravitational pull from every other body j
        for j in range(N):
            if i == j:
                continue    # idiot-proofing self-interaction

            # Compute displacement from i to j
            r_ij = bod_obj[i].r - bod_obj[j].r

            # Scalar distance squared
            dist_sq = np.sum(r_ij**2)

            # Denominator with softening factor
            denominator = (dist_sq + e**2)**1.5

            # Add the j-body contribution
            a_i += bod_obj[j].m * (r_ij / denominator)

        # Apply -G and store the net acceleration for body i
        accelerations[i] = -G * a_i

    return accelerations


In [ ]:
"""
Conservation Value Calculations

This cell contains all the functions to help with the conservation and will be further embedded into the code.
Here, we will define later-mentioned Hamiltonian and momenta conservations, as well as their relative values.
"""

# Calculate the kinetic energy of the system: T = sum_i |p_i|^2 / (2 m_i)
# bod_obj – a list of stellar bodies represented as objects.
#           Contains objects themselves. Allows to access
#           individual parameters such as: mass, initial velocity,
#           and initial displacement.
def t_abs(bod_obj):
    t = 0.0     # total kinetic energy
    # Iterate through the list of bodies to extract individual values for the velocity and mass
    for obj in bod_obj:
        p = obj.m * obj.v       # linear momentum vector of the body
        t += np.dot(p, p) / (2 * obj.m)     # |p|^2 is a dot product, so that the result is a scalar

    return t

# Calculate the (softened) gravitational potential energy of the system:
# V = -G * sum_{i<j} m_i m_j / sqrt(|r_i - r_j|^2 + e^2)
# bod_obj – a list of stellar bodies represented as objects.
# G – the constant of gravitational attraction
# e – softening factor (it has to be the same one as in acc, otherwise H is not the
#     energy that the integrator is actually conserving)
def v_abs(bod_obj, G, e):
    N = len(bod_obj)

    v_tot = 0.0     # potential energy total (without the -G factor)

    # Sum over every UNIQUE pair (i < j): a pair holds one potential energy, not two
    for i in range(N):
        for j in range(i + 1, N):
            r_ij = bod_obj[i].r - bod_obj[j].r      # separation vector
            numerator = bod_obj[i].m * bod_obj[j].m
            denominator = np.sqrt(np.dot(r_ij, r_ij) + e**2)    # |r_ij|^2 is a scalar (dot product)

            v_tot += numerator / denominator

    return -G * v_tot

# Calculate the absolute Hamiltonian value
# t – system's kinetic energy (calculated previously)
# v – system's potential energy (calculated previously)
def h_abs(t, v):
    return t + v

# Convert the Hamiltonian into a relative energy error
# h_0 – initial Hamiltonian value at the start of the simulation clock
# h_n – current Hamiltonian value at the given moment in time
def h_rel(h_0, h_n):
    return abs(h_n - h_0) / abs(h_0)

# Calculate absolute linear momentum: P = sum_i m_i v_i  (a 3-vector)
# bod_obj – a list of stellar bodies represented as objects.
def p_abs(bod_obj):
    p_total = np.zeros(3)   # initiate a total linear momentum vector

    # Iterate over every single object in the system
    for obj in bod_obj:
        p_total += obj.m * obj.v

    return p_total

# Characteristic linear momentum of the system: sum_i |m_i v_i|
# Used as the reference when the total momentum is (close to) zero.
def p_scale(bod_obj):
    return sum(obj.m * np.linalg.norm(obj.v) for obj in bod_obj)

# Calculate the Total Relative Linear Momentum Error
# p_0 – initial linear momentum vector at the start of the simulation
# p_n – current linear momentum vector at the time of the reading
# scale – characteristic momentum (p_scale at t = 0), used instead of |p_0| if |p_0| ~ 0
def p_rel(p_0, p_n, scale):
    reference = np.linalg.norm(p_0)
    if reference < 1e-12 * scale:       # P_0 = 0: relative error is undefined, use the scale
        reference = scale

    return np.linalg.norm(p_n - p_0) / reference

# Calculate Total Angular Momentum: L = sum_i m_i (r_i x v_i)  (a 3-vector)
# bod_obj – a list of stellar bodies represented as objects.
def l_abs(bod_obj):
    l_tot = np.zeros(3)     # initiate the summation vector of the total angular momentum

    # Iterate over every body in the system
    for obj in bod_obj:
        l_tot += obj.m * np.cross(obj.r, obj.v)     # cross product, NOT an element-wise product

    return l_tot

# Characteristic angular momentum of the system: sum_i |m_i (r_i x v_i)|
def l_scale(bod_obj):
    return sum(obj.m * np.linalg.norm(np.cross(obj.r, obj.v)) for obj in bod_obj)

# Calculate the Total Relative Angular Momentum Error
# l_0 – initial angular momentum vector at the start of the simulation
# l_n – current angular momentum vector
# scale – characteristic angular momentum (l_scale at t = 0), used instead of |l_0| if |l_0| ~ 0
def l_rel(l_0, l_n, scale):
    reference = np.linalg.norm(l_0)
    if reference < 1e-12 * scale:       # L_0 = 0: relative error is undefined, use the scale
        reference = scale

    return np.linalg.norm(l_n - l_0) / reference

# Snapshot of all conserved quantities at one instant: returns (H, P, L)
def conservation_state(bod_obj, G, e):
    return h_abs(t_abs(bod_obj), v_abs(bod_obj, G, e)), p_abs(bod_obj), l_abs(bod_obj)

In [ ]:
"""
Creating a Velocity Verlet 4th Order Script.
It consists of the acceleration, displacement and velocity computations.

The detailed description of the algorithm will be provided later down below.
"""

# Compute Displacement of an individual body
# obj – individual object with accessible properties
# t – current timestep
def displ_vv_4(obj, t):
    return obj.r + obj.v*t + (0.5*obj.a*(t**2))

# Compute Velocity of an individual body
# obj – individual object with accessible properties
# t – current timestep
# a_old – old acceleration of the body
def vel_vv_4(obj, t, a_old):
    return obj.v + 0.5*(a_old+obj.a)*t

# Run Velocity Verlet (4th order, Yoshida composition of three 2nd order steps)
# bod_obj – a list of stellar bodies represented as objects.
#           Contains objects themselves. Allows to access
#           individual parameters such as: mass, initial velocity,
#           and initial displacement.
# G – the constant of gravitational attraction
# e – softening factor to prevent division by 0 errors in case
#     the bodies get too close.
# dt – timestep of the simulation
# n_steps – number of timesteps to take
# acc_func – force routine used for every acceleration evaluation. Defaults to the direct
#            summation acc; any function with the same (G, bod_obj, e) arguments and output
#            can be passed instead, such as the Barnes-Hut acc_bh (see Experiment 2).
# NB: the initial conditions (r, v, m) must be set on every body before calling this.
def run_vv_4(bod_obj, G, e, dt, n_steps, acc_func=acc):
    weights = (yoshida_x1, yoshida_x0, yoshida_x1)    # sub-step sizes as fractions of dt (they sum to 1)

    # Initial accelerations of all bodies
    for obj, a in zip(bod_obj, acc_func(G, bod_obj, e)):
        obj.a = a

    # Start the clock of the simulated Universe
    for _ in range(n_steps):
        # Yoshida sub-steps: each one is a full Velocity Verlet step of size w*dt
        for w in weights:
            h = w * dt

            a_old = [obj.a for obj in bod_obj]    # accelerations before the move

            # Move ALL bodies first (new displacement)
            for obj in bod_obj:
                obj.r = displ_vv_4(obj, h)

            # Re-evaluate the forces once for the whole system (new acceleration)
            for obj, a in zip(bod_obj, acc_func(G, bod_obj, e)):
                obj.a = a

            # Update ALL velocities with the average of old and new acceleration
            for obj, a_prev in zip(bod_obj, a_old):
                obj.v = vel_vv_4(obj, h, a_prev)


## Velocity Verlet 4th Order Explanation
In order to solve the N-body problem numerically, several quantities have to be evaluated: the Euclidean distance between pairs of bodies, an initial velocity vector obtained from a randomised scalar speed, and the gravitational acceleration, which is then propagated in time by the integrator. The initial speed is scaled by a random factor in order to produce a variety of orbital eccentricities for experimental purposes.

**Notation.** Bodies are indexed by $i, j = 1, \dots, N$. Vectors carry a hat in the convention used throughout this notebook: $\hat{r}_i$ is the displacement, $\hat{v}_i$ the velocity and $\hat{a}_i$ the acceleration of body $i$, while $m_i$ is its mass. $G$ is the gravitational constant, $\delta{t}$ is the timestep, $\epsilon$ is the softening length (`e` in the code) and $\xi$ is the random eccentricity factor.

### Euclidean Distance in 3D
The separation of two bodies is the length of the difference of their displacement vectors (`euclidian_distance`, Eq. (1)):

$$r_{ij}=\lvert\hat{r}_i-\hat{r}_j\rvert=\sqrt{(x_i-x_j)^2 + (y_i - y_j)^2 + (z_i-z_j)^2} \tag{1}$$

### Initial Speed
The speed of a circular orbit at the distance $r_{ij}$ of Eq. (1) around a body of mass $M$ is $v_c=\sqrt{GM/r_{ij}}$, where $M$ is the mass of the attracting body. The initial speed is a random fraction of it (`vel_init`, Eq. (2)):

$$v_0=\xi\sqrt{\frac{GM}{r_{ij}}},\qquad \xi\sim\mathcal{U}[0,1) \tag{2}$$

The velocity is launched perpendicular to the radius vector, so the launch point is an apsis of the resulting orbit. For $\xi<1$ the body starts at apocentre and follows an ellipse of eccentricity $e_{\mathrm{orb}}=1-\xi^2$, while $\xi\to1$ gives a circular orbit. Since the escape speed $\sqrt{2}\,v_c$ is never reached, no body is ejected at the start of the simulation.

### Initial Velocity Direction
The scalar speed is converted into a vector by choosing a direction perpendicular to the radius vector $\Delta\hat{r}=\hat{r}_j-\hat{r}_i=(\Delta x,\Delta y,\Delta z)$ (`get_perpendicular` and `vel_direction`, Eq. (3)):

$$\hat{p}=(-\Delta y,\ \Delta x,\ 0),\qquad \hat{n}=\frac{\hat{p}}{\lvert\hat{p}\rvert},\qquad \hat{v}_0=v_0\,\hat{n} \tag{3}$$

The vector $\hat{p}$ is perpendicular to $\Delta\hat{r}$ because $\hat{p}\cdot\Delta\hat{r}=-\Delta x\Delta y+\Delta y\Delta x=0$. If $\Delta x=\Delta y=0$ the construction degenerates, and $\hat{p}=(0,1,0)$ is used instead. As $\hat{n}$ lies in the $xy$-plane, the initial orbits are planar.

### Gravitational Acceleration
Newton's law of gravitation, summed over all other bodies, gives the acceleration of body $i$ (`acc`, Eq. (4)). A softening length $\epsilon$ is added to the denominator (Plummer softening; Plummer, 1911; Dehnen & Read, 2011) so that the force stays finite when two bodies approach each other:

$$\hat{a}_i=-G\sum_{j\neq i}\frac{m_j\,(\hat{r}_i-\hat{r}_j)}{\left(\lvert\hat{r}_i-\hat{r}_j\rvert^2+\epsilon^2\right)^{3/2}} \tag{4}$$

This direct summation requires $\mathcal{O}(N^2)$ pair evaluations per call, which is precisely the cost that the Barnes-Hut tree reduces to $\mathcal{O}(N\log N)$ in the later C++ implementation.

### Velocity Verlet (Second Order)
Since the algorithm is a 4th-order Velocity Verlet, it integrates with a different pattern to the regular Verlet (Verlet, 1967).
Regular 2nd-order Velocity Verlet (Swope et al., 1982) uses the evaluation in a following pattern:
- Set up initial conditions
- Evaluate displacement
- Evaluate acceleration
- Evaluate velocity

I coined a name for this method, RAV, due to $\hat{r}$ being displacement, $\hat{a}$ acceleration and $\hat{v}$ velocity. Velocity Verlet can be expressed via the following equations, where a single step is denoted $S_2(\delta{t})$:

**Displacement** (`displ_vv_4`, Eq. (5))

$$\hat{r}(t+\delta{t}) = \hat{r}(t)+\hat{v}(t)\,\delta{t}+\frac{\hat{a}(t)\,\delta{t}^2}{2} \tag{5}$$

**Velocity** (`vel_vv_4`, Eq. (6))

$$\hat{v}(t+\delta{t})=\hat{v}(t)+\frac{\hat{a}(t)+\hat{a}(t+\delta{t})}{2}\,\delta{t} \tag{6}$$

The new acceleration $\hat{a}(t+\delta{t})$ in Eq. (6) is evaluated from the new displacements of Eq. (5), which is why the order of the three operations cannot be changed. The scheme is time-reversible and symplectic, but only second-order accurate: its global error scales as $\mathcal{O}(\delta{t}^2)$ (Hairer, Lubich & Wanner, 2006).

### Yoshida Composition to 4th Order
Yoshida's method (Yoshida, 1990; see also Forest & Ruth, 1990) raises the order by composing three second-order steps of different sizes (Eq. (7)):

$$S_4(\delta{t})=S_2(w_1\delta{t})\circ S_2(w_0\delta{t})\circ S_2(w_1\delta{t}) \tag{7}$$

$$w_1=\frac{1}{2-2^{1/3}}\approx 1.3512,\qquad w_0=-\frac{2^{1/3}}{2-2^{1/3}}\approx -1.7024 \tag{8}$$

In the code the weights of Eq. (8) are `yoshida_x1` and `yoshida_x0`. They are fixed by the two conditions of Eq. (9): the sub-steps must add up to one full step, and the leading error term of $S_2$ must cancel,

$$2w_1+w_0=1,\qquad 2w_1^3+w_0^3=0 \tag{9}$$

so that the local error falls from $\mathcal{O}(\delta{t}^3)$ to $\mathcal{O}(\delta{t}^5)$ and the global error to $\mathcal{O}(\delta{t}^4)$. The middle weight $w_0$ is negative, so the middle sub-step runs backwards in time, but the three sub-steps together advance the system by exactly $\delta{t}$, and the composition remains symplectic. The price is three force evaluations per timestep instead of one.

### One Integration Step in the Code
For each of the three sub-steps of size $h=w\,\delta{t}$ with $w\in\{w_1, w_0, w_1\}$ (`run_vv_4`):
1. Store the current accelerations $\hat{a}_{\mathrm{old}}$ of all bodies.
2. Move all bodies with $\delta{t}\to h$ (Eq. (5)).
3. Re-evaluate the accelerations of all bodies at the new positions (Eq. (4)).
4. Update all velocities with $\delta{t}\to h$ and $\hat{a}_{\mathrm{old}}$ (Eq. (6)).

All bodies are moved before any acceleration is recomputed, so that every force is evaluated for the same instant.

# Conservation Formulae
## Energy (Hamiltonian) Conservation

Energy conservation is a widely used metric to directly track and diagnose issues within the numerical integrators. Due to the nature of symplectic integrators, the energy error does not drift away steadily but oscillates in a bounded band around the true value, which is the signature of the symplectic property (Hairer, Lubich & Wanner, 2006). Non-symplectic methods may show a short-term success, whereas after a larger number of iterations their energy error grows systematically.

### Absolute Energy Conservation
Kinetic energy $T$ is expressed via

$$T=\sum^N_{i=1}\frac{|\hat{p}_i|^2}{2m_i} \tag{10}$$

while potential energy $V$ is a sum over every *unique* pair of bodies:

$$V=-G\sum_{1\leq i < j \leq N} \frac{m_i m_j}{\sqrt{|\hat{r}_i - \hat{r}_j|^2 + \epsilon^2}} \tag{11}$$

where $\hat{p}_i = m_i \hat{v}_i$ and $|\hat{r}_i - \hat{r}_j|$ is the Euclidean distance between two bodies, Eq. (1). The strict inequality $i<j$ matters: a pair holds a single potential energy, so summing over $i\neq j$ (or $i\leq j$) would double-count it (or include self-interaction). The softening length $\epsilon$ is the same as in the acceleration, since $\hat{a}_i=-\frac{1}{m_i}\nabla_{\hat{r}_i}V$ reproduces Eq. (4) exactly.

Hence a Hamiltonian $H$ can be derived as the sum of Eq. (10) and Eq. (11):
$$
H(\hat{r}, \hat{p}) = T + V = \sum^N_{i=1}\frac{|\hat{p}_i|^2}{2m_i} - G\sum_{1\leq i < j \leq N} \frac{m_i m_j}{\sqrt{|\hat{r}_i - \hat{r}_j|^2 + \epsilon^2}} \tag{12}
$$

### Relative Energy Conservation $H_{rel}$
Relative energy is the main tracking tool for the energy conservation, since it normalises absolute values to fit a relative parameter. It is expressed like this, where $H_{n}$ is the Hamiltonian of Eq. (12) evaluated at step $n$ and $H_{0}$ is its initial value:
$$
H_{rel} = \frac{|H_{n}-H_{0}|}{|H_{0}|} \tag{13}
$$

## Momentum Calculus
Momentum calculations for both Linear $\hat{P}$ and Angular $\hat{L}$ momenta are required, since the momenta have to stay constant alongside with the oscillating or near-constant energy. Meaning, deviations from the constant may indicate underlying issues with the integrators or the simulation loops. All of the above-mentioned parameters allow us to track accuracy.

### Total Linear Momentum $\hat{P}$
Total linear momentum is simply the vector sum of individual linear momenta across all bodies and expressed like this:
$$
\hat{P}=\sum^N_{i=1}m_i\hat{v}_i \tag{14}
$$

It is a 3-vector, hence each component from $x$ to $z$ inclusively is summed separately.

### Relative Total Linear Momentum Conservation $P_{rel}$
Relative total linear momentum is one of the main tracking tools for the momenta conservation, since it normalises absolute values to fit a relative parameter. Since $\hat{P}$ (Eq. (14)) is a vector, its *length* is used:
$$
P_{rel} = \frac{|\hat{P}_{n}-\hat{P}_{0}|}{|\hat{P}_{0}|} \tag{15}
$$

Systems that are set up in their centre-of-mass frame (such as the figure-eight of Experiment 1; Moore, 1993; Chenciner & Montgomery, 2000) have $\hat{P}_0=0$, for which Eq. (15) is undefined. In that case the denominator is replaced by the characteristic momentum of the system, $\sum_i m_i|\hat{v}_i(0)|$, so that $P_{rel}$ remains the error relative to the size of the individual momenta.

### Angular Momentum $\hat{L}$
$$
\hat{L} = \sum^N_{i=1}m_i(\hat{r}_i\times\hat{v}_i) \tag{16}
$$

The angular momentum is the other part of the momenta conservation toolkit. It is a vector built from the **cross product** of displacement and velocity (not the product of their unit vectors, and not a component-wise product). Both linear and angular momenta are conserved by the exact dynamics, and for pairwise central forces a symplectic integrator preserves them to rounding error (Hairer, Lubich & Wanner, 2006), which makes them a sharp test for coding mistakes in the force routine and the loop.

### Relative Angular Momentum Conservation Expression $L_{rel}$
As with the prior equations, the relative angular momentum error is computed via the length of the difference between the initial and current angular momenta (Eq. (16)), divided by the length of the initial one:

$$
L_{rel} = \frac{|\hat{L}_{n}-\hat{L}_{0}|}{|\hat{L}_{0}|} \tag{17}
$$

The figure-eight also has $\hat{L}_0=0$, so the denominator of Eq. (17) is again replaced by the characteristic value $\sum_i m_i|\hat{r}_i(0)\times\hat{v}_i(0)|$.

In [ ]:
"""
Experiment 1: the three-body figure-eight choreography (28/10/26).

Purpose:
    The figure-eight is a periodic solution of the planar three-body problem in which
    three equal masses chase one another along a single closed curve shaped like the
    number eight. It was found numerically by Moore (1993) and its existence was proven
    by Chenciner & Montgomery (2000). Because the exact answer is known (the bodies must
    return to their starting positions after every period T), it is an ideal first
    validation test for the integrator: any drift away from the starting positions after
    a whole number of periods is a direct measure of the numerical error.

Setup:
    - three equal masses chasing each other along a single figure-eight curve
    - initial conditions from Simo's high-precision values, in natural units G = m = 1
    - no softening: the bodies never come close, and softening would perturb the orbit
    - integrated with run_vv_4 (Velocity Verlet 4th order)

Time control:
    - the total simulated time is a whole number of orbital periods
    - dt is dictated by the number of timesteps: dt = t_total / n_steps
    - frames are computed separately: one trajectory point is stored every
      steps_per_frame = n_steps / n_frames steps
Conservation tracking:
    The Hamiltonian H and the linear and angular momenta P and L are evaluated at every stored
    frame (conservation_state), and their relative errors H_rel, P_rel and L_rel are plotted
    together with the orbit in 3D and its 2D projection onto the x-y plane. The output is these five
    plots plus the closure error after the run.
"""


# A minimal body: mass, displacement, velocity and acceleration.
# The integrator (run_vv_4) and the force routine (acc) only ever access the
# attributes m, r, v and a, so any object that carries these four attributes
# can be simulated. All vectors are stored as 3-component NumPy arrays, so that
# the same code works for planar (z = 0) and fully three-dimensional problems.
class Body:
    def __init__(self, m, r, v):
        self.m = m      # mass (scalar)
        self.r = np.array(r, dtype=float)       # displacement vector (x, y, z)
        self.v = np.array(v, dtype=float)       # velocity vector (vx, vy, vz)
        self.a = np.zeros(3)        # acceleration vector, filled in by run_vv_4 on its first call


# Experiment parameters (natural units: G = 1, m = 1)
# The published figure-eight initial conditions are dimensionless: they assume G = 1
# and unit masses. Working in SI units would require rescaling every quantity, so the
# experiment is run directly in these natural units. Lengths, times and velocities
# printed below are therefore dimensionless as well.
G_nat = 1.0     # gravitational constant in natural units

# Plummer softening (Eq. 4) is switched off. The closest approach between any two
# bodies on the figure-eight is about 0.7 length units, so there is no risk of a
# division by zero, and any non-zero softening would alter the force law and hence
# destroy the periodicity that this experiment relies on.
e_soft = 0.0    # softening length (switched off, see above)

# Period of one full figure-eight orbit in natural units. After exactly one period
# every body returns to its own starting position. The value is only known to a
# finite number of digits, which sets a lower floor on the closure error below.
T_period = 6.32591398     # period of the figure-eight orbit (Simo's value, 9 significant digits)

# The three numbers below fully control the time discretisation:
#   n_periods – how long the simulation runs (t_total = n_periods * T_period)
#   n_steps – how finely that time is divided (this alone dictates dt)
#   n_frames – how many snapshots are stored for plotting (independent of dt)
# Increasing n_steps makes dt smaller and the result more accurate (the global error
# of the 4th-order scheme falls as dt^4) at a proportionally higher cost, while
# n_frames only changes the resolution of the plots and the memory used.
n_periods = 3   # number of full orbits to simulate
n_steps = 30000     # total number of timesteps (dictates dt)
n_frames = 1500     # number of stored trajectory points (independent of dt)

# Derived time quantities
t_total = n_periods * T_period      # total simulated time
dt = t_total / n_steps      # timestep implied by the chosen number of steps

# The integration is split into n_frames equal blocks of steps_per_frame steps each,
# and one snapshot is stored at the end of every block. For the blocks to add up to
# exactly n_steps, n_steps must be an integer multiple of n_frames; otherwise the run
# would stop short of (or overshoot) t_total and the closure test would be invalid.
assert n_steps % n_frames == 0, "n_steps must be divisible by n_frames"
steps_per_frame = n_steps // n_frames       # integration steps between two stored snapshots

# Initial conditions
# The configuration at t = 0 is symmetric:
#   – body 2 sits exactly at the crossing point (centre) of the eight
#   – bodies 1 and 3 sit at mirror-image positions, r3 = -r1
#   – bodies 1 and 3 share the same velocity, and body 2 moves with -2 times it
# These conditions give zero total momentum (m*v1 + m*v2 + m*v3 = 0), so the centre of
# mass stays fixed at the origin and no correction for drift is needed (unlike in a
# system with a randomly launched star). All z-components are zero, so the motion
# remains in the x-y plane for the entire run.
r1 = [0.97000436, -0.24308753, 0.0]     # displacement of body 1 (body 3 is its mirror image)
v2 = [-0.93240737, -0.86473146, 0.0]    # velocity of the central body 2
v13 = [-0.5 * v2[0], -0.5 * v2[1], 0.0]     # v1 = v3 = -v2 / 2, so the total momentum is zero

# The three bodies, each with unit mass
bodies = [
    Body(1.0, r1, v13),     # body 1: right-hand lobe
    Body(1.0, [0.0, 0.0, 0.0], v2),     # body 2: centre of the eight
    Body(1.0, [-r1[0], -r1[1], 0.0], v13),      # body 3: left-hand lobe
]

# Copies of the starting positions, kept for the closure test at the end of the run.
# .copy() is essential: without it r_start would refer to the same arrays that the
# integrator overwrites, and the closure error would always read zero.
r_start = [b.r.copy() for b in bodies]

# Summary of the time discretisation actually used
print(f"t_total = {t_total:.4f} ({n_periods} periods), n_steps = {n_steps}, dt = {dt:.3e}")
print(f"n_frames = {n_frames}, steps_per_frame = {steps_per_frame}")

# Conservation references
# The state at t = 0 is the reference for every relative error. P_0 and L_0 are exactly
# zero for this configuration (see the initial-conditions comment above), so p_rel and
# l_rel fall back on the characteristic size of the individual momenta, p_scale and l_scale.
H_0, P_0, L_0 = conservation_state(bodies, G_nat, e_soft)
P_scale, L_scale = p_scale(bodies), l_scale(bodies)
print(f"H_0 = {H_0:.6f}, |P_0| = {np.linalg.norm(P_0):.1e}, |L_0| = {np.linalg.norm(L_0):.1e}")

# Simulation
# trajectories[k] is the list of stored displacements of body k. It starts with the
# initial position, so after the loop each list holds n_frames + 1 points (t = 0 plus
# one point per block). run_vv_4 advances the bodies in place by steps_per_frame steps
# of size dt; after each call the new positions are copied into the record and the
# conserved quantities are measured on the freshly updated bodies.
# NB: run_vv_4 recomputes the accelerations at the start of every call. This costs one
# extra force evaluation per frame but does not change the result, because the
# accelerations depend only on the current positions.
trajectories = [[b.r.copy()] for b in bodies]
h_err, p_err, l_err = [0.0], [0.0], [0.0]       # relative errors at t = 0 are zero by definition
for _ in range(n_frames):
    run_vv_4(bodies, G_nat, e_soft, dt, steps_per_frame)
    for k, b in enumerate(bodies):
        trajectories[k].append(b.r.copy())

    H_n, P_n, L_n = conservation_state(bodies, G_nat, e_soft)
    h_err.append(h_rel(H_0, H_n))
    p_err.append(p_rel(P_0, P_n, P_scale))
    l_err.append(l_rel(L_0, L_n, L_scale))

# Convert the nested lists into a single array for convenient slicing:
#   axis 0 – body index k, axis 1 -> frame index, axis 2 -> coordinate (x, y, z)
trajectories = np.array(trajectories)       # shape: (N bodies, n_frames + 1, 3)
h_err, p_err, l_err = np.array(h_err), np.array(p_err), np.array(l_err)
t_frames = np.arange(n_frames + 1) * steps_per_frame * dt / T_period    # time in orbital periods

# Validation
# Closure check: after a whole number of periods every body should be back where it
# started. The distance |r(t_total) - r(0)| therefore measures the accumulated error.
# It contains two contributions: the integration error of the scheme (which shrinks as
# dt^4 when n_steps is increased) and the finite precision of T_period and of the
# initial conditions (which does not shrink with dt). Once the error stops decreasing
# when n_steps is raised, the second contribution has become dominant.
for k, b in enumerate(bodies):
    print(f"body {k + 1}: closure error |r(T) - r(0)| = {np.linalg.norm(b.r - r_start[k]):.3e}")

# Planarity check: with all z-components initially zero, the accelerations of Eq. (4)
# have no z-component either, so z must remain exactly zero for the whole run.
print(f"max |z| over the run = {np.abs(trajectories[:, :, 2]).max():.3e} (the orbit is planar)")

# Conservation summary
print(f"max H_rel = {h_err.max():.3e}, max P_rel = {p_err.max():.3e}, max L_rel = {l_err.max():.3e}")

# Plots: 3D orbit + 2D orbit + the three relative conservation errors
colours = plt.cm.tab10(np.arange(3))        # one distinct colour per body

# All three bodies travel along the very same curve, so lines of equal width would hide
# one another. Drawing them with decreasing widths (body 1 widest, body 3 thinnest)
# keeps every trajectory visible as a coloured band inside the previous one.
line_widths = [3.0, 1.8, 0.8]
fig = plt.figure(figsize=(16, 11))

# Layout on a 2 x 6 grid: the two orbit views share the top row (3 columns each) and the
# three conservation panels share the bottom row (2 columns each).
grid = fig.add_gridspec(2, 6)

# 3D view
# The orbit is planar, so this view shows the eight lying flat at z = 0. Any spurious
# out-of-plane motion would be visible here. The end markers (o) should lie on top of
# the start markers (x) because the run covers a whole number of periods.
ax3d = fig.add_subplot(grid[0, 0:3], projection="3d")
for k in range(len(bodies)):
    x, y, z = trajectories[k].T     # transpose (frames, 3) three coordinate arrays
    ax3d.plot(x, y, z, color=colours[k], lw=line_widths[k], label=f"Body {k + 1}")
    ax3d.scatter(x[0], y[0], z[0], color=colours[k], marker="x", s=30)     # starting position
    ax3d.scatter(x[-1], y[-1], z[-1], color=colours[k], s=30)       # final position

# Use the same limits on all three axes and a cubic box, so that lengths in different
# directions are directly comparable and the orbit is not visually distorted.
lim = 1.1 * np.abs(trajectories).max()      # 10% margin around the furthest point reached
ax3d.set_xlim(-lim, lim)
ax3d.set_ylim(-lim, lim)
ax3d.set_zlim(-lim, lim)
ax3d.set_box_aspect((1, 1, 1))
ax3d.set_xlabel("x")
ax3d.set_ylabel("y")
ax3d.set_zlabel("z")
ax3d.set_title("Figure-eight orbit in 3D (x = start, o = end)")
ax3d.legend(fontsize=8, loc="upper left")

# 2D view
# The motion is planar, so the projection onto the x-y plane shows the true shape of the
# orbit without the perspective distortion of the 3D view. An equal aspect ratio keeps the
# eight undistorted, so that its two lobes can be compared directly.
ax2d = fig.add_subplot(grid[0, 3:6])
for k in range(len(bodies)):
    x, y = trajectories[k, :, 0], trajectories[k, :, 1]
    ax2d.plot(x, y, color=colours[k], lw=line_widths[k], label=f"Body {k + 1}")
    ax2d.scatter(x[0], y[0], color=colours[k], marker="x", s=40, zorder=3)     # starting position
    ax2d.scatter(x[-1], y[-1], color=colours[k], s=40, zorder=3)       # final position
ax2d.set_xlim(-lim, lim)
ax2d.set_ylim(-lim, lim)
ax2d.set_aspect("equal")
ax2d.set_xlabel("x")
ax2d.set_ylabel("y")
ax2d.set_title("Figure-eight orbit in the x-y plane (x = start, o = end)")
ax2d.legend(fontsize=8, loc="upper left")
ax2d.grid(alpha=0.3)

# Relative conservation errors (logarithmic y-axis, time measured in orbital periods).
# The floor of 1e-18 only keeps exact zeros (e.g. at t = 0) drawable on a log axis.
floor = 1e-18
panels = [
    (h_err, r"$H_{rel}=|H_n-H_0|/|H_0|$", "Relative Hamiltonian error", "tab:blue"),
    (p_err, r"$P_{rel}=|\hat{P}_n-\hat{P}_0|/\sum_i m_i|\hat{v}_i(0)|$", "Relative linear momentum error", "tab:green"),
    (l_err, r"$L_{rel}=|\hat{L}_n-\hat{L}_0|/\sum_i m_i|\hat{r}_i(0)\times\hat{v}_i(0)|$", "Relative angular momentum error", "tab:red"),
]
for idx, (err, ylabel, title, colour) in enumerate(panels):
    ax = fig.add_subplot(grid[1, 2 * idx:2 * idx + 2])
    ax.semilogy(t_frames, np.maximum(err, floor), color=colour, lw=1.2)
    ax.set_xlabel("time (orbital periods)")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(alpha=0.3, which="both")

fig.suptitle(f"Three-body figure-eight: {n_periods} periods, {n_steps} steps, "
             f"dt = {dt:.2e} (Velocity Verlet 4th order)")
plt.tight_layout()
plt.show()

## Creating the Barnes-Hut Octree
The direct summation of Eq. (4) evaluates every pair of bodies and therefore scales as $\mathcal{O}(N^2)$. The cell below implements the Barnes-Hut octree (Barnes & Hut, 1986), which approximates the same acceleration in $\mathcal{O}(N\log N)$ operations. The routine `acc_bh` is written as a drop-in replacement for `acc`, and `run_vv_4` accepts either of them through its `acc_func` argument, so that the integrator itself remains unchanged.


In [ ]:
"""
Creating the Barnes-Hut Octree (30/10/26).
It consists of the tree construction, the mass moments of every node and the tree walk
that evaluates the approximate gravitational acceleration of every body.

acc_bh has exactly the same signature as the direct summation acc (plus the opening
angle theta), so it can be handed to run_vv_4 through its acc_func argument.
The detailed description of the algorithm will be provided later down below.
"""

# Maximum depth of the tree. Two bodies sitting at (almost) the same position can never be
# separated by subdividing the cube, so without this limit the insertion would recurse
# forever. At depth 32 a cube is 2^32 times smaller than the root, far below any physical
# separation in these simulations, so the limit only ever acts as a safety net.
max_depth_bh = 32


# A single node of the octree: one cubic cell of space.
# centre – geometric centre of the cube (3-vector)
# half – half of the side length of the cube (scalar)
# m – total mass of all bodies inside the cube (filled in by moments_bh)
# com – centre of mass of all bodies inside the cube (filled in by moments_bh)
# bodies – indices of the bodies stored in this node (only used while it is a leaf)
# children – list of 8 child nodes once the cube has been subdivided, None for a leaf
class Node:
    def __init__(self, centre, half):
        self.centre = np.array(centre, dtype=float)
        self.half = half
        self.m = 0.0
        self.com = np.zeros(3)
        self.bodies = []
        self.children = None


# Find the octant (0-7) of a node that contains the displacement r.
# Each bit of the index answers one question: bit 0 – is x on the positive side of the
# centre, bit 1 – is y, bit 2 – is z. Hence octant = [x >= c_x] + 2[y >= c_y] + 4[z >= c_z].
# node – the node being subdivided
# r – displacement vector of the body
def octant_bh(node, r):
    return int(r[0] >= node.centre[0]) + 2 * int(r[1] >= node.centre[1]) + 4 * int(r[2] >= node.centre[2])


# Split a leaf into 8 equal child cubes of half the side length.
# The child with index k is shifted from the parent centre by half/2 in every direction,
# with the sign of each shift taken from the corresponding bit of k (same rule as octant_bh).
# node – the leaf to be subdivided
def subdivide_bh(node):
    quarter = 0.5 * node.half       # half-width of every child
    node.children = []
    for k in range(8):
        sign = np.array([1.0 if k & 1 else -1.0,
                         1.0 if k & 2 else -1.0,
                         1.0 if k & 4 else -1.0])
        node.children.append(Node(node.centre + quarter * sign, quarter))


# Insert body i into the subtree that starts at node.
# A leaf may hold a single body. When a second body arrives the leaf is subdivided, the
# body already living there is pushed down into its octant, and the insertion continues
# into the octant of the new body. This repeats until every body sits alone in a leaf.
# node – current node of the descent
# i – index of the body in bod_obj
# bod_obj – a list of stellar bodies represented as objects
# depth – depth of node in the tree (the root has depth 0)
def insert_bh(node, i, bod_obj, depth):
    if node.children is None:
        # Empty leaf (or the depth limit is reached): simply store the body here
        if not node.bodies or depth >= max_depth_bh:
            node.bodies.append(i)
            return

        # Occupied leaf: subdivide and push the resident body one level down
        subdivide_bh(node)
        for j in node.bodies:
            insert_bh(node.children[octant_bh(node, bod_obj[j].r)], j, bod_obj, depth + 1)
        node.bodies = []

    # Internal node: descend into the octant that contains body i
    insert_bh(node.children[octant_bh(node, bod_obj[i].r)], i, bod_obj, depth + 1)


# Compute the total mass and the centre of mass of every node (post-order traversal).
# Leaf: M = sum of the masses of its bodies, R = their mass-weighted mean position.
# Internal node: the same sums taken over the 8 children, M = sum_k M_k, R = sum_k M_k R_k / M.
# node – root of the subtree
# bod_obj – a list of stellar bodies represented as objects
def moments_bh(node, bod_obj):
    if node.children is None:
        for j in node.bodies:
            node.m += bod_obj[j].m
            node.com += bod_obj[j].m * bod_obj[j].r
    else:
        for child in node.children:
            moments_bh(child, bod_obj)
            node.m += child.m
            node.com += child.m * child.com

    if node.m > 0.0:
        node.com /= node.m      # numerator above is sum m r, divide by the mass to obtain R


# Build the complete octree for the current positions of all bodies.
# The root is the smallest cube that encloses every body, enlarged by a tiny margin so that
# no body lies exactly on its outer boundary.
# bod_obj – a list of stellar bodies represented as objects
def build_tree_bh(bod_obj):
    positions = np.array([obj.r for obj in bod_obj])
    lower, upper = positions.min(axis=0), positions.max(axis=0)
    centre = 0.5 * (lower + upper)
    half = 0.5 * np.max(upper - lower) * (1.0 + 1e-6) + 1e-12    # +1e-12 guards a single body

    root = Node(centre, half)
    for i in range(len(bod_obj)):
        insert_bh(root, i, bod_obj, 0)
    moments_bh(root, bod_obj)

    return root


# Check whether the displacement r lies inside the cube of a node
def contains_bh(node, r):
    return np.all(np.abs(r - node.centre) <= node.half)


# Acceleration of body i due to all the mass inside the subtree that starts at node.
# Opening criterion: a node of side s = 2*half seen from distance d = |r_i - R| is accepted
# as a single point mass at its centre of mass if s / d < theta, otherwise it is opened and
# its children are visited. A node that contains body i itself is always opened (see the
# explanation below). Leaves are summed exactly, which is Eq. (4) restricted to the leaf.
# node – current node of the tree walk
# i – index of the body that feels the force
# bod_obj – a list of stellar bodies represented as objects
# G – the constant of gravitational attraction
# e – softening factor (same Plummer softening as in acc)
# theta – opening angle
def node_acc_bh(node, i, bod_obj, G, e, theta):
    a_i = np.zeros(3)
    if node.m == 0.0:
        return a_i      # empty cube: no contribution

    r_i = bod_obj[i].r

    # Leaf: exact summation over the bodies stored in it (skipping self-interaction)
    if node.children is None:
        for j in node.bodies:
            if j == i:
                continue
            r_ij = r_i - bod_obj[j].r
            denominator = (np.dot(r_ij, r_ij) + e**2)**1.5
            a_i += -G * bod_obj[j].m * r_ij / denominator
        return a_i

    # Internal node: accept it as one pseudo-body or open it
    r_iN = r_i - node.com       # displacement from the centre of mass of the node to body i
    dist = np.sqrt(np.dot(r_iN, r_iN))
    s = 2.0 * node.half         # side length of the cube

    # s / d < theta is written without a division, and the (more expensive) containment test is
    # only evaluated for nodes that already pass the opening criterion
    if s < theta * dist and not contains_bh(node, r_i):
        denominator = (dist**2 + e**2)**1.5
        return -G * node.m * r_iN / denominator

    for child in node.children:
        a_i += node_acc_bh(child, i, bod_obj, G, e, theta)

    return a_i


# Compute Gravitational Acceleration of All Bodies with the Barnes-Hut approximation.
# A drop-in replacement for acc: same arguments and same output (a list of 3-vectors).
# G – the constant of gravitational attraction
# bod_obj – a list of stellar bodies represented as objects
# e – softening factor
# theta – opening angle; theta = 0 opens every node and reproduces direct summation,
#         larger values are faster but less accurate (0.5 - 1.0 is the usual range)
def acc_bh(G, bod_obj, e, theta=0.5):
    root = build_tree_bh(bod_obj)       # the tree is rebuilt at every force evaluation, since the bodies move
    return [node_acc_bh(root, i, bod_obj, G, e, theta) for i in range(len(bod_obj))]


## Barnes-Hut Octree Explanation
The cost of a gravitational N-body simulation is dominated by the force evaluation. Eq. (4) sums the contribution of every body $j\neq i$ to the acceleration of body $i$, which requires $N(N-1)$ pair evaluations per call, and the 4th-order Velocity Verlet calls it three times per timestep. For the large $N$ of cosmological applications this quadratic cost is prohibitive.

The Barnes-Hut algorithm (Barnes & Hut, 1986) rests on a simple physical observation: a compact group of bodies that is far away from body $i$ pulls on it almost exactly like a single body carrying the total mass of the group and located at its centre of mass. The algorithm organises the bodies into a hierarchy of cubic cells, an *octree*, and decides for every cell whether it is distant enough to be replaced by such a pseudo-body or whether it has to be examined in more detail.

**Notation.** In addition to the notation of the Velocity Verlet explanation, $c$ denotes a cell (a node of the tree), $\hat{c}$ its geometric centre, $s$ its side length (`2 * half` in the code), $M_c$ the total mass it contains and $\hat{R}_c$ its centre of mass. $d$ is the distance from body $i$ to $\hat{R}_c$ and $\theta$ is the opening angle.

### Spatial Decomposition (Octree Construction)
The root cell is the smallest cube that encloses all bodies (`build_tree_bh`). Every cell can be split into eight equal octants of half the side length (`subdivide_bh`). The octant of a cell that contains a displacement $\hat{r}=(x,y,z)$ is identified by three binary comparisons with the centre $\hat{c}=(c_x,c_y,c_z)$ (`octant_bh`, Eq. (18)):

$$k=[x\geq c_x]+2\,[y\geq c_y]+4\,[z\geq c_z],\qquad k\in\{0,\dots,7\} \tag{18}$$

where $[\,\cdot\,]$ equals 1 if the condition holds and 0 otherwise. The centre of child $k$ is displaced from the parent centre by a quarter of the parent side length along each axis (Eq. (19)),

$$\hat{c}_k=\hat{c}+\frac{s}{4}\,(\sigma_x,\sigma_y,\sigma_z),\qquad \sigma\in\{-1,+1\} \tag{19}$$

with each sign $\sigma$ taken from the corresponding bit of $k$.

The bodies are inserted one by one (`insert_bh`). A body descends from the root into the octant that contains it until it reaches an empty leaf, where it is stored. When it reaches a leaf that is already occupied, that leaf is subdivided and both bodies are pushed further down, until each of them occupies a leaf of its own. For a roughly uniform distribution the depth of the tree grows as $\log_8 N$, so the construction costs $\mathcal{O}(N\log N)$. Two bodies at (almost) identical positions cannot be separated by any number of subdivisions, so a maximum depth (`max_depth_bh`) is imposed, beyond which a leaf may hold more than one body.

### Mass Moments of the Cells
Every cell is described by its total mass and its centre of mass (`moments_bh`, Eq. (20)):

$$M_c=\sum_{j\in c}m_j,\qquad \hat{R}_c=\frac{1}{M_c}\sum_{j\in c}m_j\,\hat{r}_j \tag{20}$$

These are computed from the bottom of the tree upwards: a leaf sums over its bodies directly, while an internal cell combines the moments of its eight children, $M_c=\sum_k M_k$ and $\hat{R}_c=\sum_k M_k\hat{R}_k/M_c$. Every body therefore contributes to one cell per level, and the whole pass costs $\mathcal{O}(N\log N)$.

### Opening Criterion and Tree Walk
The acceleration of body $i$ is obtained by walking the tree from the root (`node_acc_bh`). For each cell the ratio of its size to its distance from body $i$ is compared with the opening angle,

$$\frac{s}{d}<\theta,\qquad d=\lvert\hat{r}_i-\hat{R}_c\rvert \tag{21}$$

If the criterion of Eq. (21) holds, the cell subtends a small angle as seen from body $i$ and its entire content is replaced by a single softened point mass at $\hat{R}_c$ (Eq. (22)):

$$\hat{a}_i^{(c)}=-G\,\frac{M_c\,(\hat{r}_i-\hat{R}_c)}{\left(\lvert\hat{r}_i-\hat{R}_c\rvert^2+\epsilon^2\right)^{3/2}} \tag{22}$$

which has exactly the form of a single term of Eq. (4). If the criterion fails, the cell is *opened* and the same test is applied to each of its children. When a leaf is reached, its bodies are summed exactly as in Eq. (4). The total acceleration is the sum of all accepted contributions (`acc_bh`).

A cell that contains body $i$ itself is always opened, regardless of the criterion. Without this rule a large cell whose centre of mass happens to lie far from body $i$ could pass the test although body $i$ sits inside it, which produces large force errors (Salmon & Warren, 1994). The rule also guarantees that body $i$ never interacts with itself, since it can only be met inside its own leaf, where the term $j=i$ is skipped.

The opening angle controls the trade-off between accuracy and cost:
- $\theta=0$: no cell is ever accepted, every cell is opened down to its leaves, and the algorithm reproduces direct summation exactly (up to the order of floating-point additions). This limit is used in Experiment 2 to validate the implementation.
- $\theta\approx0.5$–$1$: the range commonly used in practice.
- larger $\theta$: faster, but increasingly inaccurate.

### Accuracy and Cost
Because the expansion is taken about the centre of mass, the dipole term of the multipole expansion of a cell vanishes identically. The leading error of the point-mass approximation is therefore the quadrupole term (Salmon & Warren, 1994), and the relative error of a single accepted cell scales as

$$\frac{\lvert\delta\hat{a}_i^{(c)}\rvert}{\lvert\hat{a}_i^{(c)}\rvert}\sim\mathcal{O}\!\left(\left(\frac{s}{d}\right)^2\right)\lesssim\mathcal{O}(\theta^2) \tag{23}$$

so, by Eq. (23), halving $\theta$ reduces the force error roughly fourfold. The number of cells accepted by a single body grows only logarithmically with $N$ (approximately as $\theta^{-3}\log N$), hence one complete force evaluation costs $\mathcal{O}(N\log N)$ instead of $\mathcal{O}(N^2)$ (Barnes & Hut, 1986). The tree has to be rebuilt at every force evaluation, since the bodies move, but this is also an $\mathcal{O}(N\log N)$ operation.

In this pure-Python prototype the bookkeeping of the tree outweighs the saving for the small $N$ that can be simulated, so `acc_bh` is not yet faster than `acc`. The asymptotic advantage becomes decisive only for large $N$ in the compiled C++ implementation.

### Consequences for the Conserved Quantities
The approximation does not only introduce a small force error; it also breaks the symmetries on which the conservation laws of the previous section rely.

**Linear momentum.** In direct summation the force of $j$ on $i$ is exactly opposite to the force of $i$ on $j$ (Newton's third law), so the internal forces cancel in pairs and $\hat{P}$ (Eq. (14)) is conserved. In the tree, body $i$ may see body $j$ as part of a distant cell, while body $j$ sees body $i$ individually or as part of a different cell. The two forces are then no longer equal and opposite, and $\hat{P}$ drifts (Springel, 2005).

**Angular momentum.** For the same reason, and because an accepted cell pulls towards $\hat{R}_c$ rather than along the individual lines joining the bodies, the internal torques no longer cancel and $\hat{L}$ (Eq. (16)) is not exactly conserved either.

**Energy.** Velocity Verlet and its Yoshida composition are symplectic for forces that derive from a potential (Hairer, Lubich & Wanner, 2006), $\hat{a}_i=-\frac{1}{m_i}\nabla_{\hat{r}_i}V$. The tree force is not the gradient of the exact potential $V$ of the Hamiltonian, nor of any single smooth potential, since the tree is rebuilt at every evaluation and the force changes discontinuously whenever a cell switches between being accepted and being opened. The bounded energy oscillation that characterises a symplectic integrator is therefore no longer guaranteed, and $H_{rel}$ (Eq. (13)), which is always evaluated with the exact $V$ (Eq. (11)), is expected to be larger and to grow with $\theta$.

Measuring how strongly these three properties degrade for a given $\theta$ is precisely the purpose of Experiment 2.

### One Force Evaluation in the Code
1. Find the bounding cube of all bodies and insert them one by one into the octree (`build_tree_bh`, `insert_bh`, `subdivide_bh`, `octant_bh`).
2. Compute $M_c$ and $\hat{R}_c$ for every cell from the leaves upwards (`moments_bh`).
3. For every body $i$, walk the tree from the root, accepting cells that satisfy $s/d<\theta$ and do not contain body $i$, and opening all others (`node_acc_bh`, `contains_bh`).
4. Return the list of accelerations in the same format as `acc` (`acc_bh`), so that `run_vv_4(..., acc_func=...)` can use it without modification.


In [ ]:
"""
Experiment 2: Barnes-Hut octree against direct summation in a Plummer star cluster (30/10/26).

Purpose:
    Experiment 1 validated the Velocity Verlet 4th order integrator on a problem with a
    known exact answer. This experiment keeps the integrator fixed and changes only the
    force routine, in order to isolate the effect of the Barnes-Hut approximation on the
    reliability of the simulation. The same initial conditions are integrated with the
    same timestep three times:
        - direct summation, acc (the reference, exact up to rounding)
        - Barnes-Hut, acc_bh with theta = 0.5 (the usual accuracy setting)
        - Barnes-Hut, acc_bh with theta = 1.0 (a deliberately coarse setting)
    Any difference between the runs is therefore caused by the force approximation alone.

Setup:
    - N equal-mass bodies drawn from a Plummer sphere (Plummer 1911), the standard
      self-gravitating test model of N-body dynamics, sampled with the method of
      Aarseth, Henon & Wielen (1974)
    - Henon units: G = 1, total mass M = 1, total energy E = -1/4, which fixes the
      Plummer scale length to a = 3*pi/16
    - the cluster is moved to its centre-of-mass frame and its velocities are rescaled to
      exact virial equilibrium (2T = -V), so it should neither collapse nor expand
    - Plummer softening is switched ON: the bodies are randomly placed, so close encounters
      do happen, and the softening keeps the force finite (Eq. 4)
    - the system is fully three-dimensional, so all eight octants of the tree are used

Time control:
    Identical to Experiment 1: t_total is a whole number of crossing times, dt = t_total / n_steps,
    and one snapshot is stored every steps_per_frame = n_steps / n_frames steps.

Conservation tracking:
    H, P and L are evaluated at every stored frame with the EXACT expressions of the
    conservation cell (direct pair sums), never with the tree. The Barnes-Hut runs are thus
    judged against the true Hamiltonian, which is the quantity that matters physically.

Expected outcome:
    - direct summation: P and L conserved to rounding error, H_rel small and bounded
    - Barnes-Hut: P and L no longer exact (tree forces violate Newton's third law), and
      H_rel larger, growing with theta (tree forces are not the gradient of the exact V)
    - individual trajectories of the three runs agree at first and then separate, because
      the N-body problem is chaotic and amplifies any small force difference exponentially
"""


# Plummer sphere sampler (Aarseth, Henon & Wielen 1974)
# Draws N bodies of mass M / N whose positions follow the Plummer density profile
#   rho(r) = 3 M / (4 pi a^3) * (1 + r^2 / a^2)^(-5/2)
# and whose speeds follow the matching isotropic distribution function.
# N – number of bodies
# G – the constant of gravitational attraction
# M – total mass of the cluster
# a – Plummer scale length
# r_cut – radius cut-off in units of a (removes the very rare, extremely distant bodies)
def init_plummer(N, G, M, a, r_cut=10.0):
    bod_obj = []

    # Random unit vector, uniformly distributed over the surface of a sphere
    def random_direction():
        cos_t = 2.0 * rnd.random() - 1.0        # cos(polar angle) uniform in [-1, 1]
        phi = 2.0 * np.pi * rnd.random()        # azimuth uniform in [0, 2 pi)
        sin_t = np.sqrt(1.0 - cos_t**2)
        return np.array([sin_t * np.cos(phi), sin_t * np.sin(phi), cos_t])

    for _ in range(N):
        # Radius: invert the cumulative mass profile M(r) / M = X  ->  r = a / sqrt(X^(-2/3) - 1)
        while True:
            X = rnd.random()
            if X > 0.0:
                r = a / np.sqrt(X**(-2.0 / 3.0) - 1.0)
                if r < r_cut * a:
                    break

        # Speed: v = q * v_esc, with q drawn from g(q) = q^2 (1 - q^2)^(7/2) by rejection.
        # The maximum of g on [0, 1] is about 0.092, so a box of height 0.1 encloses it.
        while True:
            q, y = rnd.random(), 0.1 * rnd.random()
            if y < q**2 * (1.0 - q**2)**3.5:
                break
        v_esc = np.sqrt(2.0 * G * M / np.sqrt(r**2 + a**2))     # local escape speed of the Plummer potential

        bod_obj.append(Body(M / N, r * random_direction(), q * v_esc * random_direction()))

    # Move to the centre-of-mass frame: subtract the mass-weighted mean position and velocity,
    # so that the cluster does not drift and P_0 = 0 (up to rounding)
    r_cm = sum(obj.m * obj.r for obj in bod_obj) / M
    v_cm = sum(obj.m * obj.v for obj in bod_obj) / M
    for obj in bod_obj:
        obj.r = obj.r - r_cm
        obj.v = obj.v - v_cm

    return bod_obj


# Rescale all velocities so that the cluster is in exact virial equilibrium, 2T + V = 0.
# Random sampling of N bodies only satisfies this on average; the rescaling removes the
# sampling noise so that the experiment starts from a genuinely equilibrated cluster.
def virialise(bod_obj, G, e):
    t, v = t_abs(bod_obj), v_abs(bod_obj, G, e)
    factor = np.sqrt(-0.5 * v / t)      # T scales with the square of the velocities
    for obj in bod_obj:
        obj.v = obj.v * factor


# Independent copy of a list of bodies, so that every run starts from identical conditions.
# Body() wraps r and v in np.array, which copies them.
def copy_bodies(bod_obj):
    return [Body(obj.m, obj.r, obj.v) for obj in bod_obj]


# Integrate one copy of the initial conditions and record trajectories and conservation errors.
# The loop is the same as in Experiment 1, wrapped in a function so that it can be repeated
# for every force routine. Only the time spent inside run_vv_4 is measured as the runtime.
# bod_init – initial conditions (left untouched)
# acc_func – force routine handed to run_vv_4
def simulate(bod_init, G, e, dt, n_frames, steps_per_frame, acc_func):
    bod_obj = copy_bodies(bod_init)

    H_0, P_0, L_0 = conservation_state(bod_obj, G, e)
    P_sc, L_sc = p_scale(bod_obj), l_scale(bod_obj)

    trajectories = [[obj.r.copy()] for obj in bod_obj]
    h_err, p_err, l_err = [0.0], [0.0], [0.0]
    runtime = 0.0

    for _ in range(n_frames):
        clock = time.perf_counter()
        run_vv_4(bod_obj, G, e, dt, steps_per_frame, acc_func)
        runtime += time.perf_counter() - clock

        for k, obj in enumerate(bod_obj):
            trajectories[k].append(obj.r.copy())

        H_n, P_n, L_n = conservation_state(bod_obj, G, e)
        h_err.append(h_rel(H_0, H_n))
        p_err.append(p_rel(P_0, P_n, P_sc))
        l_err.append(l_rel(L_0, L_n, L_sc))

    return {"traj": np.array(trajectories), "h": np.array(h_err), "p": np.array(p_err),
            "l": np.array(l_err), "time": runtime}


# Experiment parameters (Henon units: G = 1, M = 1, E = -1/4)
G_nat = 1.0     # gravitational constant
M_tot = 1.0     # total mass of the cluster
a_pl = 3.0 * np.pi / 16.0       # Plummer scale length giving E = -1/4 in Henon units
N_bodies = 64       # number of bodies (kept small: the prototype is pure Python)

# Softening: a few per cent of the scale length. It must be large enough to tame close
# encounters at the chosen dt, and small compared with the mean separation of the bodies
# (about 0.4 length units here) so that the cluster still behaves like point masses.
e_soft_2 = 0.05

# The crossing time t_cr = 2 sqrt(2) is the time a typical body needs to cross the cluster
# in Henon units. Two crossing times are long enough for the bodies to complete roughly
# one orbit and for the runs to separate, while keeping the pure-Python runtime modest.
t_cross = 2.0 * np.sqrt(2.0)
n_crossings = 2     # simulated time in crossing times
n_steps_2 = 500     # total number of timesteps (dictates dt)
n_frames_2 = 250    # number of stored snapshots (independent of dt)

t_total_2 = n_crossings * t_cross
dt_2 = t_total_2 / n_steps_2
assert n_steps_2 % n_frames_2 == 0, "n_steps_2 must be divisible by n_frames_2"
steps_per_frame_2 = n_steps_2 // n_frames_2

# Force routines under test. acc_bh takes the opening angle as an extra argument, so a
# lambda fixes theta and leaves the (G, bod_obj, e) signature that run_vv_4 expects.
theta_values = [0.5, 1.0]
force_routines = {"Direct summation": acc}
for th in theta_values:
    force_routines[f"Barnes-Hut theta = {th}"] = (lambda G, b, e, th=th: acc_bh(G, b, e, th))
    # th=th binds the current value; a plain lambda would see only the last theta of the loop

# Initial conditions (re-seeded here, so the cluster is the same regardless of which cells ran before)
rnd.seed(seed)
cluster = init_plummer(N_bodies, G_nat, M_tot, a_pl)
virialise(cluster, G_nat, e_soft_2)

print(f"N = {N_bodies}, t_total = {t_total_2:.4f} ({n_crossings} crossing times), "
      f"n_steps = {n_steps_2}, dt = {dt_2:.3e}, softening = {e_soft_2}")
H_init = h_abs(t_abs(cluster), v_abs(cluster, G_nat, e_soft_2))
print(f"H_0 = {H_init:.5f} (-0.25 for an infinite Plummer sphere; the difference is sampling noise of finite N), "
      f"virial ratio 2T/|V| = {2 * t_abs(cluster) / abs(v_abs(cluster, G_nat, e_soft_2)):.3f}")

# Force accuracy at t = 0
# Before integrating, the tree accelerations are compared body by body with direct summation:
# delta_i = |a_i(BH) - a_i(direct)| / |a_i(direct)|. theta = 0 must reproduce direct summation
# to rounding error, which validates the tree construction and the walk.
a_direct = acc(G_nat, cluster, e_soft_2)
print("\nForce accuracy at t = 0 (relative error per body):")
for th in [0.0] + theta_values:
    a_tree = acc_bh(G_nat, cluster, e_soft_2, th)
    delta = np.array([np.linalg.norm(at - ad) / np.linalg.norm(ad) for at, ad in zip(a_tree, a_direct)])
    print(f"  theta = {th:.1f}: mean = {delta.mean():.2e}, max = {delta.max():.2e}")

# Simulations (the same initial conditions, three force routines)
print("\nRunning simulations (this takes a few minutes in pure Python)...")
results = {}
for label, func in force_routines.items():
    results[label] = simulate(cluster, G_nat, e_soft_2, dt_2, n_frames_2, steps_per_frame_2, func)
    res = results[label]
    print(f"  {label:<24} runtime = {res['time']:6.1f} s, "
          f"max H_rel = {res['h'].max():.2e}, max P_rel = {res['p'].max():.2e}, max L_rel = {res['l'].max():.2e}")

t_frames_2 = np.arange(n_frames_2 + 1) * steps_per_frame_2 * dt_2 / t_cross     # time in crossing times

# Divergence from the reference: mean distance between the same body in a tree run and in the
# direct run at the end of the simulation, in units of the Plummer scale length.
ref_label = "Direct summation"
for label, res in results.items():
    if label != ref_label:
        sep = np.linalg.norm(res["traj"][:, -1] - results[ref_label]["traj"][:, -1], axis=1).mean()
        print(f"  mean final separation from the direct run ({label}) = {sep / a_pl:.2e} a")

# Plots: 3D and 2D trajectories + the three relative conservation errors
# The trajectories shown are those of the theta = 0.5 run. With 64 bodies a full plot would be
# unreadable, so a handful of bodies that stay inside the cluster (r < 2.5 a for the whole run)
# are drawn in colour, while the final positions of all bodies are shown as grey dots.
show_label = f"Barnes-Hut theta = {theta_values[0]}"
traj_show = results[show_label]["traj"]
traj_ref = results[ref_label]["traj"]

n_show = 6
bound = [k for k in range(N_bodies) if np.linalg.norm(traj_show[k], axis=1).max() < 2.5 * a_pl]
highlight = rnd.sample(bound, min(n_show, len(bound)))
colours_2 = plt.cm.tab10(np.arange(len(highlight)))
lim_2 = 1.1 * np.abs(traj_show[highlight]).max()

fig = plt.figure(figsize=(16, 11))
grid = fig.add_gridspec(2, 6)

# 3D view of the highlighted orbits
ax3d = fig.add_subplot(grid[0, 0:3], projection="3d")
final = traj_show[:, -1]
inside = np.all(np.abs(final) < lim_2, axis=1)      # 3D axes do not clip, so far-away bodies are left out
ax3d.scatter(*final[inside].T, color="grey", s=6, alpha=0.5, label="all bodies (final)")
for c, k in enumerate(highlight):
    x, y, z = traj_show[k].T
    ax3d.plot(x, y, z, color=colours_2[c], lw=1.2)
    ax3d.scatter(x[0], y[0], z[0], color=colours_2[c], marker="x", s=30)
    ax3d.scatter(x[-1], y[-1], z[-1], color=colours_2[c], s=30)
ax3d.set_xlim(-lim_2, lim_2)
ax3d.set_ylim(-lim_2, lim_2)
ax3d.set_zlim(-lim_2, lim_2)
ax3d.set_box_aspect((1, 1, 1))
ax3d.set_xlabel("x")
ax3d.set_ylabel("y")
ax3d.set_zlabel("z")
ax3d.set_title(rf"Orbits in 3D, Barnes-Hut $\theta$ = {theta_values[0]} (x = start, o = end)")
ax3d.legend(fontsize=8, loc="upper left")

# 2D projection onto the x-y plane, with the direct-summation orbit of the same body dashed.
# The two curves start on top of each other and separate as the chaotic dynamics amplify the
# small force differences: this is the visual counterpart of the final-separation printout.
ax2d = fig.add_subplot(grid[0, 3:6])
ax2d.scatter(final[:, 0], final[:, 1], color="grey", s=6, alpha=0.5)
for c, k in enumerate(highlight):
    ax2d.plot(traj_show[k, :, 0], traj_show[k, :, 1], color=colours_2[c], lw=1.4)
    ax2d.plot(traj_ref[k, :, 0], traj_ref[k, :, 1], color=colours_2[c], lw=0.9, ls="--")
    ax2d.scatter(traj_show[k, 0, 0], traj_show[k, 0, 1], color=colours_2[c], marker="x", s=30)
    ax2d.scatter(traj_show[k, -1, 0], traj_show[k, -1, 1], color=colours_2[c], s=30)
ax2d.plot([], [], color="black", lw=1.4, label=rf"Barnes-Hut $\theta$ = {theta_values[0]}")
ax2d.plot([], [], color="black", lw=0.9, ls="--", label="Direct summation")
ax2d.set_xlim(-lim_2, lim_2)
ax2d.set_ylim(-lim_2, lim_2)
ax2d.set_aspect("equal")
ax2d.set_xlabel("x")
ax2d.set_ylabel("y")
ax2d.set_title("Orbits projected onto the x-y plane")
ax2d.legend(fontsize=8, loc="upper left")
ax2d.grid(alpha=0.3)

# Relative conservation errors of all three runs on common axes
floor = 1e-18
run_colours = {ref_label: "black"}
for th, colour in zip(theta_values, ["tab:blue", "tab:orange"]):
    run_colours[f"Barnes-Hut theta = {th}"] = colour

panels_2 = [
    ("h", r"$H_{rel}$", "Relative Hamiltonian error"),
    ("p", r"$P_{rel}$ (relative to $\sum_i m_i|\hat{v}_i(0)|$)", "Relative linear momentum error"),
    ("l", r"$L_{rel}$", "Relative angular momentum error"),
]
for idx, (key, ylabel, title) in enumerate(panels_2):
    ax = fig.add_subplot(grid[1, 2 * idx:2 * idx + 2])
    for label, res in results.items():
        ax.semilogy(t_frames_2, np.maximum(res[key], floor), color=run_colours[label], lw=1.1,
                    label=label.replace("theta", r"$\theta$"))     # LaTeX theta in the legend
    ax.set_xlabel("time (crossing times)")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(alpha=0.3, which="both")
    ax.legend(fontsize=8)

fig.suptitle(f"Plummer cluster, N = {N_bodies}: {n_crossings} crossing times, {n_steps_2} steps, "
             f"dt = {dt_2:.2e} (Velocity Verlet 4th order, direct summation vs Barnes-Hut)")
plt.tight_layout()
plt.show()


# Creating Leapfrog Integrator

## About Leapfrog

The leapfrog method is a second-order symplectic integrator and a close relative of the Velocity Verlet integrator; in the kick-drift-kick (KDK) form applied here the two are, in fact, algebraically identical (see the explanation below). The name refers to the staggered structure of the scheme: the velocities are evaluated half a timestep out of phase with the displacements, so that the two quantities "leap" over one another in turn. The implementation reuses the helper functions and the gravitational acceleration of Eq. (1) to Eq. (4).

Within each timestep the algorithm proceeds in four stages:
- **Kick:** advance the velocities by half a timestep with the current acceleration $\hat{a}_n$
- **Drift:** advance the displacements by a full timestep with the half-step velocities
- **Force evaluation:** compute the new accelerations $\hat{a}_{n+1}$ at the new displacements
- **Kick:** advance the velocities by the remaining half timestep with $\hat{a}_{n+1}$

In [ ]:
"""
Leapfrog Second-Order Implementation.
Consists of the displacement, velocity and runtime computations.

Detailed algorithmic solution will be provided later down below in the description.
"""

# Compute initial kick by half a timestep
# obj – individual object with accessible properties
# t – current timestep
# Theoretically, the initial half-velocity will be assigned to the object and retrieved later
def veli_l_2(obj, t):
    return obj.v + obj.a * (0.5 * t)

# Compute Displacement applying first kick
# obj – individual object with accessible properties
# t – current timestep
def displ_l_2(obj, t):
    return obj.r + obj.v * t

# Compute final kick by the second half-step
# obj – individual object with accessible properties
# t – current timestep
def velf_l_2(obj, t):
    return obj.v + obj.a * (0.5 * t)

# Run Leapfrog
# bod_obj – a list of stellar bodies represented as objects.
#           Contains objects themselves. Allows to access
#           individual parameters such as: mass, initial velocity,
#           and initial displacement.
# G – the constant of gravitational attraction
# e – softening factor to prevent division by 0 errors in case
#     the bodies get too close.
# dt – timestep of the simulation
# n_steps – number of timesteps to take
# acc_func – force routine used for every acceleration evaluation. Defaults to the direct
#            summation acc; any function with the same (G, bod_obj, e) arguments and output
#            can be passed instead, such as the Barnes-Hut acc_bh (see Experiment 2).
# NB: the initial conditions (r, v, m) must be set on every body before calling this.
def run_l_2(bod_obj, G, e, dt, n_steps, acc_func=acc):

    # Find initial accelerations of all bodies
    for obj, a in zip(bod_obj, acc_func(G, bod_obj, e)):
        obj.a = a

    # Start the clock of the simulated Universe
    for _ in range(n_steps):

        # Kick: half-step velocity of ALL bodies with the old acceleration a_n
        for obj in bod_obj:
            obj.v = veli_l_2(obj, dt)

        # Drift: move ALL bodies by a full step with the half-step velocity
        for obj in bod_obj:
            obj.r = displ_l_2(obj, dt)

        # Re-evaluate the forces once for the whole system (new acceleration a_{n+1})
        for obj, a in zip(bod_obj, acc_func(G, bod_obj, e)):
            obj.a = a

        # Kick: complete the velocity update with the new acceleration a_{n+1}
        for obj in bod_obj:
            obj.v = velf_l_2(obj, dt)


## Leapfrog 2nd Order Explanation
The leapfrog integrator is a second-order method in its own right and therefore requires no Yoshida composition. It advances the system with the kick-drift-kick (KDK) sequence introduced above, in which a *kick* changes only the velocities and a *drift* changes only the displacements.

**Notation.** In this section the subscript $n$ denotes the timestep, so that $\hat{r}_n$, $\hat{v}_n$ and $\hat{a}_n$ are the displacement, velocity and acceleration of a body at the time $t_n=n\,\delta{t}$, while $\hat{v}_{n+1/2}$ is its velocity at the intermediate time $t_n+\delta{t}/2$. The body index $i$ is omitted for readability; every equation applies to all bodies simultaneously.

### Update Equations
**First Kick** (`veli_l_2`, Eq. (24))

$$\hat{v}_{n+1/2}=\hat{v}_n+\frac{1}{2}\,\hat{a}_n\,\delta{t} \tag{24}$$

**Drift** (`displ_l_2`, Eq. (25))

$$\hat{r}_{n+1}=\hat{r}_n+\hat{v}_{n+1/2}\,\delta{t} \tag{25}$$

The new acceleration $\hat{a}_{n+1}$ is then obtained from Eq. (4) at the displacements $\hat{r}_{n+1}$, and the velocity update is completed by a second half kick:

**Second Kick** (`velf_l_2`, Eq. (26))

$$\hat{v}_{n+1}=\hat{v}_{n+1/2}+\frac{1}{2}\,\hat{a}_{n+1}\,\delta{t} \tag{26}$$

The functions `veli_l_2` and `velf_l_2` contain the same expression. They differ only in the acceleration stored on the body at the moment they are called, which is $\hat{a}_n$ for the first kick and $\hat{a}_{n+1}$ for the second.

### Kick and Drift Operators
The step can be written compactly in terms of two elementary maps: the kick $K(h)$, which sets $\hat{v}\leftarrow\hat{v}+\hat{a}(\hat{r})\,h$ at fixed $\hat{r}$, and the drift $D(h)$, which sets $\hat{r}\leftarrow\hat{r}+\hat{v}\,h$ at fixed $\hat{v}$. One leapfrog step is then (Eq. (27))

$$S_2^{\mathrm{L}}(\delta{t})=K\!\left(\tfrac{\delta{t}}{2}\right)\circ D(\delta{t})\circ K\!\left(\tfrac{\delta{t}}{2}\right) \tag{27}$$

where the maps are applied from right to left (the sequence is symmetric, so it reads the same in either direction). The drift is the exact solution of the equations of motion for the kinetic part $T$ of the Hamiltonian of Eq. (12) alone, and the kick is the exact solution for the potential part $V$ alone. The exact flow of any Hamiltonian is symplectic, and a composition of symplectic maps is again symplectic; hence the leapfrog step is symplectic. Because the sequence is symmetric, the step is also time-reversible, $S_2^{\mathrm{L}}(-\delta{t})\circ S_2^{\mathrm{L}}(\delta{t})=\mathrm{id}$. Its local error is $\mathcal{O}(\delta{t}^3)$ and its global error $\mathcal{O}(\delta{t}^2)$, and, as for every symplectic method, the energy error remains bounded and oscillatory instead of drifting (Hairer, Lubich & Wanner, 2006).

### Relation to Velocity Verlet
Substituting Eq. (24) into Eq. (25) gives $\hat{r}_{n+1}=\hat{r}_n+\hat{v}_n\,\delta{t}+\frac{1}{2}\hat{a}_n\,\delta{t}^2$, which is Eq. (5), and adding Eq. (24) and Eq. (26) gives $\hat{v}_{n+1}=\hat{v}_n+\frac{1}{2}(\hat{a}_n+\hat{a}_{n+1})\,\delta{t}$, which is Eq. (6). The KDK leapfrog is therefore algebraically identical to the second-order Velocity Verlet step $S_2$ (Swope et al., 1982). The two differ only in the order of the floating-point operations, so that their results agree to rounding error.

### Cost and the Staggered Form
A single force evaluation is required per timestep: the acceleration $\hat{a}_{n+1}$ computed for the second kick is stored on the body and reused by the first kick of the following step. Moreover, when the velocities are not required at whole timesteps, the second kick of one step and the first kick of the next can be merged into a single full kick (Eq. (28)),

$$\hat{v}_{n+1/2}=\hat{v}_{n-1/2}+\hat{a}_n\,\delta{t} \tag{28}$$

which is the classical staggered form that gives the method its name: the displacements are defined at whole timesteps and the velocities at half timesteps, each leaping over the other. The implementation retains the synchronised KDK form, since the Hamiltonian and the momenta of Eq. (12) to Eq. (17) require displacements and velocities at the same instant.

### One Integration Step in the Code
Before the first step, `run_l_2` evaluates the initial accelerations $\hat{a}_0$ of all bodies (Eq. (4)). Each timestep then consists of:
1. Kick all velocities by half a timestep with $\hat{a}_n$ (Eq. (24)).
2. Drift all displacements by a full timestep (Eq. (25)).
3. Re-evaluate the accelerations $\hat{a}_{n+1}$ of all bodies at the new displacements (Eq. (4)).
4. Kick all velocities by the remaining half timestep with $\hat{a}_{n+1}$ (Eq. (26)).

All bodies are moved before any acceleration is recomputed, so that every force is evaluated for the same instant.

# Forest-Ruth 4th Order Integrator (Forest and Ruth, 1990)

## About

The Forest-Ruth 4th order integrator (FR4) is an explicit symplectic integrator for separable Hamiltonians of the form $H=T(\hat{p})+V(\hat{r})$, such as Eq. (12), and therefore exhibits no long-term energy drift. It is implemented here in the form in which Forest and Ruth (1990) originally formulated it: as a sequence of four kick-drift stages, each with its own pair of coefficients, rather than as a composition of second-order steps. It is nevertheless mathematically identical to the Velocity Verlet 4th order of Eq. (7); the two differ only in the way in which they are composed (see the explanation below).

FR4 follows this pattern within each timestep:
- Stage 1: kick-drift operation giving $\hat{v}_1$ and $\hat{r}_1$, followed by the acceleration $\hat{a}_1=\hat{a}(\hat{r}_1)$
- Stage 2: kick-drift operation giving $\hat{v}_2$ and $\hat{r}_2$, followed by $\hat{a}_2=\hat{a}(\hat{r}_2)$
- Stage 3: kick-drift operation giving $\hat{v}_3$ and $\hat{r}_3$, followed by $\hat{a}_3=\hat{a}(\hat{r}_3)$
- Stage 4: final kick $\hat{v}_4=\hat{v}_3+c_4\,\delta{t}\,\hat{a}_3$. There is no drift in this stage, since $d_4=0$.

In [ ]:
"""
Forest-Ruth 4th Order Implementation (Forest & Ruth, 1990).
Consists of the coefficients, the kick (velocity) and drift (displacement) sub-steps and the runtime computation.

The integrator follows the original kick-drift recurrence of Forest and Ruth (1990) with their
coefficient set in which d_4 = 0. The detailed description of the algorithm will be provided later down below.
"""

# Real root of the cubic 48 chi^3 + 24 chi^2 - 1 = 0 (called x by Forest & Ruth, 1990)
fr_chi = (2**(1/3) + 2**(-1/3) - 1) / 6

# Kick coefficients c_1..c_4 and drift coefficients d_1..d_4 (each set sums to 1)
fr_c = (fr_chi + 0.5, -fr_chi, -fr_chi, fr_chi + 0.5)
fr_d = (2*fr_chi + 1, -4*fr_chi - 1, 2*fr_chi + 1, 0.0)

# Compute the kick of an individual body (velocity update)
# obj – individual object with accessible properties
# t – sub-step size, c_k * dt
def vel_fr_4(obj, t):
    return obj.v + obj.a * t

# Compute the drift of an individual body (displacement update)
# obj – individual object with accessible properties
# t – sub-step size, d_k * dt
def displ_fr_4(obj, t):
    return obj.r + obj.v * t

# Run Forest-Ruth (4th order)
# bod_obj – a list of stellar bodies represented as objects.
#           Contains objects themselves. Allows to access
#           individual parameters such as: mass, initial velocity,
#           and initial displacement.
# G – the constant of gravitational attraction
# e – softening factor to prevent division by 0 errors in case
#     the bodies get too close.
# dt – timestep of the simulation
# n_steps – number of timesteps to take
# acc_func – force routine used for every acceleration evaluation. Defaults to the direct
#            summation acc; any function with the same (G, bod_obj, e) arguments and output
#            can be passed instead, such as the Barnes-Hut acc_bh (see Experiment 2).
# NB: the initial conditions (r, v, m) must be set on every body before calling this.
def run_fr_4(bod_obj, G, e, dt, n_steps, acc_func=acc):

    # Find initial accelerations of all bodies
    for obj, a in zip(bod_obj, acc_func(G, bod_obj, e)):
        obj.a = a

    # Start the clock of the simulated Universe
    for _ in range(n_steps):

        # Four kick-drift stages with the coefficients (c_k, d_k)
        for c, d in zip(fr_c, fr_d):

            # Kick: update ALL velocities with the current acceleration
            for obj in bod_obj:
                obj.v = vel_fr_4(obj, c * dt)

            # Stage 4 has d_4 = 0: the bodies do not move, so the current accelerations
            # stay valid and are reused by the first kick of the next timestep
            if d == 0.0:
                continue

            # Drift: move ALL bodies with the freshly kicked velocities
            for obj in bod_obj:
                obj.r = displ_fr_4(obj, d * dt)

            # Re-evaluate the forces once for the whole system at the new displacements
            for obj, a in zip(bod_obj, acc_func(G, bod_obj, e)):
                obj.a = a


## Forest-Ruth 4th Order Explanation
Forest and Ruth (1990) extended the kick-drift idea of the leapfrog method to fourth order. Instead of a single symmetric kick-drift-kick sequence, a timestep is split into four stages, each consisting of one kick and one drift of a different size. The sizes are chosen so that the error terms of the individual stages cancel up to third order.

**Notation.** As in the leapfrog explanation, the body index is omitted and every equation applies to all bodies simultaneously. The subscript $k=0,\dots,4$ now counts the stages within a single timestep: $\hat{r}_0$ and $\hat{v}_0$ are the displacement and velocity at the start of the step, $\hat{r}_4$ and $\hat{v}_4$ are those at its end, $t+\delta{t}$, and $\hat{a}_k=\hat{a}(\hat{r}_k)$ is given by Eq. (4).

### The Kick-Drift Recurrence
Each stage $k=1,\dots,4$ consists of a kick with the coefficient $c_k$ followed by a drift with the coefficient $d_k$ (`vel_fr_4` and `displ_fr_4`, Eq. (29)):

$$\hat{v}_k=\hat{v}_{k-1}+c_k\,\hat{a}_{k-1}\,\delta{t},\qquad \hat{r}_k=\hat{r}_{k-1}+d_k\,\hat{v}_k\,\delta{t} \tag{29}$$

The drift uses the velocity produced by the kick of the same stage, so the velocities of a stage must always be updated before its displacements (Forest & Ruth, 1990).

### Coefficients
The coefficients are expressed through $\chi$, the real root of the cubic of Eq. (30). Forest and Ruth (1990) denote this root $x$; the symbol $\chi$ is used here to avoid a clash with the coordinate $x$.

$$48\chi^3+24\chi^2-1=0,\qquad \chi=\frac{2^{1/3}+2^{-1/3}-1}{6}\approx 0.1756 \tag{30}$$

$$c_1=c_4=\chi+\tfrac{1}{2},\qquad c_2=c_3=-\chi,\qquad d_1=d_3=2\chi+1,\qquad d_2=-4\chi-1,\qquad d_4=0 \tag{31}$$

Numerically, $c_1\approx 0.6756$, $c_2\approx -0.1756$, $d_1\approx 1.3512$ and $d_2\approx -1.7024$. In the code they are `fr_c` and `fr_d`. Both sets sum to unity (Eq. (32)),

$$\sum_{k=1}^{4}c_k=\sum_{k=1}^{4}d_k=1 \tag{32}$$

so that the kicks and the drifts each advance the system by exactly one timestep in total; the remaining order conditions reduce to the cubic of Eq. (30). As a result the local error is $\mathcal{O}(\delta{t}^5)$ and the global error $\mathcal{O}(\delta{t}^4)$. Three of the coefficients are negative, so parts of every timestep run backwards in time, and $|d_2|>1$, so the middle drift overshoots a full timestep. These large coefficients give the method a comparatively large error constant for a fourth-order scheme, a well-known drawback that later optimised variants seek to reduce (Omelyan, Mryglod & Folk, 2002).

Since $d_4=0$, the fourth stage is a kick only, and with the kick and drift operators of Eq. (27) one full step reads (Eq. (33))

$$S_4^{\mathrm{FR}}(\delta{t})=K(c_4\delta{t})\circ D(d_3\delta{t})\circ K(c_3\delta{t})\circ D(d_2\delta{t})\circ K(c_2\delta{t})\circ D(d_1\delta{t})\circ K(c_1\delta{t}) \tag{33}$$

with the maps applied from right to left. The sequence is symmetric ($c_k=c_{5-k}$ and $d_k=d_{4-k}$), so the method is time-reversible, and as a composition of exact kicks and drifts it is symplectic.

### Relation to the Yoshida Composition
The coefficients of Eq. (31) are related to the Yoshida weights of Eq. (8) by $d_1=d_3=w_1$, $d_2=w_0$, $c_1=c_4=w_1/2$ and $c_2=c_3=(w_1+w_0)/2$. Writing each second-order step of Eq. (7) as the leapfrog step of Eq. (27) and merging the adjacent half kicks, $K(\tfrac{w_1}{2}\delta{t})\circ K(\tfrac{w_0}{2}\delta{t})=K(\tfrac{w_1+w_0}{2}\delta{t})$, reproduces Eq. (33) exactly. Since the KDK leapfrog is identical to Velocity Verlet, FR4 in this form is mathematically identical to the Velocity Verlet 4th order of `run_vv_4`: the two compute the same trajectory and differ only by rounding error. Forest and Ruth arrived at the method directly from the order conditions, while Yoshida (1990) obtained it, and integrators of arbitrary even order, by composing second-order steps. Forest and Ruth (1990) also give an equivalent coefficient set with $c_1=0$, which starts every step with a drift; it corresponds to the Yoshida composition of the drift-kick-drift leapfrog and is a distinct fourth-order method of the same cost.

### One Integration Step in the Code
Before the first step, `run_fr_4` evaluates the initial accelerations $\hat{a}_0$ of all bodies (Eq. (4)). Each timestep then consists of four stages $k=1,\dots,4$:
1. Kick all velocities by $c_k\,\delta{t}$ with the current accelerations (Eq. (29)).
2. If $d_k\neq 0$, drift all displacements by $d_k\,\delta{t}$ (Eq. (29)).
3. If $d_k\neq 0$, re-evaluate the accelerations of all bodies at the new displacements (Eq. (4)).

In the fourth stage $d_4=0$, so the displacements are left unchanged and the accelerations $\hat{a}_3$ remain valid for the first kick of the following step. The method therefore requires three force evaluations per timestep, the same number as `run_vv_4`. All bodies are moved before any acceleration is recomputed, so that every force is evaluated for the same instant.

# Non-Symplectic Integrators
---

## About
Non-symplectic integration is the opposite of the previously encountered symplectic methods. They show wonders in short-term simulations, especially notable methods like RK4. However, as soon as the simulation obtains longer periods, the energy drift, linear and angular momenta, and the geometric properties of the trajectories are not conserved anymore. Hence why, the simulation runs compared between symplectic and non-symplectic methods may have different trajectories and data, even though the problem setup may be the same.

# Forward Euler

Forward Euler is the simplest of all integration methods, since it has only one order of operations and updates acceleration, displacement and velocity in this pattern. Through pure theory and practical applications this method was proven to be the simplest one to write, but even short-term property conservation is less than desired.

In [ ]:
"""
Forward Euler Implementation.
Includes the displacement, velocity and summation.
"""

# Displacement Calculation
# obj – individual object with accessible properties
# t – current timestep
# NB: this uses the OLD velocity v_n, so it has to be called before the velocities are updated.
#     With the opposite order of the two updates the method becomes the symplectic Euler method.
def displ_fe_1(obj, t):
    return obj.r + (obj.v*t)

# Velocity Calculation
# obj – individual object with accessible properties
# t – current timestep
def vel_fe_1(obj, t):
    return obj.v + (t * obj.a)

# Simulation Runtime
# bod_obj – a list of stellar bodies represented as objects.
#           Contains objects themselves. Allows to access
#           individual parameters such as: mass, initial velocity,
#           and initial displacement.
# G – the constant of gravitational attraction
# e – softening factor to prevent division by 0 errors in case
#     the bodies get too close.
# dt – timestep of the simulation
# n_steps – number of timesteps to take
# acc_func – force routine used for every acceleration evaluation. Defaults to the direct
#            summation acc; any function with the same (G, bod_obj, e) arguments and output
#            can be passed instead, such as the Barnes-Hut acc_bh (see Experiment 2).
# NB: the initial conditions (r, v, m) must be set on every body before calling this.
def run_fe_1(bod_obj, G, e, dt, n_steps, acc_func=acc):

    # Find initial accelerations of all bodies
    for obj, a in zip(bod_obj, acc_func(G, bod_obj, e)):
        obj.a = a

    # Start the clock of the simulated Universe
    for _ in range(n_steps):

        # Move ALL bodies first, using the old velocity v_n
        for obj in bod_obj:
            obj.r = displ_fe_1(obj, dt)

        # Update ALL velocities with the old acceleration a_n (still stored from the previous step)
        for obj in bod_obj:
            obj.v = vel_fe_1(obj, dt)

        # Re-evaluate the forces once for the whole system at the new displacements (a_{n+1}),
        # so that the stored accelerations always belong to the stored displacements
        for obj, a in zip(bod_obj, acc_func(G, bod_obj, e)):
            obj.a = a


## Forward Euler 1st Order Explanation
The forward (explicit) Euler method is the simplest of all one-step integrators: it advances the system along the tangent to the solution at the beginning of the timestep and ignores how that tangent changes during the step. It is only first-order accurate and, unlike the integrators of the previous sections, it is neither symplectic nor time-reversible. It requires a single force evaluation per timestep and is included here as the baseline of the non-symplectic comparison.

**Notation.** As with the earlier explanation, the subscript $n$ denotes the timestep, so that $\hat{r}_n$, $\hat{v}_n$ and $\hat{a}_n=\hat{a}(\hat{r}_n)$ are the displacement, velocity and acceleration of a body at the time $t_n=n\,\delta{t}$, with $\hat{a}$ given by Eq. (4). The body index $i$ is omitted for readability and restored only where a sum over the bodies is required. The kick $K(h)$ and the drift $D(h)$ are the elementary maps introduced for Eq. (27).

### Update Equations
The equations of motion form a first-order system, $\dot{\hat{r}}=\hat{v}$ and $\dot{\hat{v}}=\hat{a}(\hat{r})$. Forward Euler replaces each time derivative by a forward difference over one timestep:

**Displacement** (`displ_fe_1`, Eq. (34))

$$\hat{r}_{n+1}=\hat{r}_n+\hat{v}_n\,\delta{t} \tag{34}$$

**Velocity** (`vel_fe_1`, Eq. (35))

$$\hat{v}_{n+1}=\hat{v}_n+\hat{a}_n\,\delta{t} \tag{35}$$

Both right-hand sides contain only quantities at the time $t_n$, so no information from the end of the step is used. In the code this fixes the order of the operations: the displacements must be updated before the velocities, so that Eq. (34) still receives $\hat{v}_n$. If the velocities were updated first, Eq. (34) would receive $\hat{v}_{n+1}$ instead, and the scheme would become the symplectic Euler method discussed below.

### Order of Accuracy
Expanding the exact solution in a Taylor series about $t_n$ and subtracting Eq. (34) and Eq. (35) gives the error of a single step (Eq. (36)),

$$\hat{r}(t_{n+1})-\hat{r}_{n+1}=\tfrac{1}{2}\hat{a}_n\,\delta{t}^2+\mathcal{O}(\delta{t}^3),\qquad \hat{v}(t_{n+1})-\hat{v}_{n+1}=\tfrac{1}{2}\dot{\hat{a}}_n\,\delta{t}^2+\mathcal{O}(\delta{t}^3) \tag{36}$$

where the exact solution is started from $\hat{r}_n$ and $\hat{v}_n$. The displacement error is exactly the quadratic term that Eq. (5) of the Velocity Verlet method retains. The local error is therefore $\mathcal{O}(\delta{t}^2)$, and since $\mathcal{O}(1/\delta{t})$ steps are required to cover a fixed time interval, the global error is only $\mathcal{O}(\delta{t})$: halving the timestep halves the error (Hairer, Lubich & Wanner, 2006). This is the lowest order of all integrators in this notebook, while the cost per step is the same single force evaluation as for the leapfrog, which is second-order.

### Loss of the Geometric Properties
**Symplecticity.** The simplest setting in which this can be seen is a body of unit mass in one dimension with a potential $V(q)$, so that $\dot{q}=p$ and $\dot{p}=-V'(q)$. One Euler step maps $(q_n,p_n)$ to $(q_n+\delta{t}\,p_n,\ p_n-\delta{t}\,V'(q_n))$ with the Jacobian determinant (Eq. (37))

$$\det\frac{\partial(q_{n+1},p_{n+1})}{\partial(q_n,p_n)}=\det\begin{pmatrix}1&\delta{t}\\-\delta{t}\,V''(q_n)&1\end{pmatrix}=1+\delta{t}^2\,V''(q_n) \tag{37}$$

A symplectic map with one degree of freedom preserves the phase-space area and therefore has the determinant one (Hairer, Lubich & Wanner, 2006). Wherever $V''\neq0$ Eq. (37) differs from one, so the Euler step is not symplectic: the area is expanded where $V''>0$ and contracted where $V''<0$. For the harmonic oscillator $V=\tfrac{1}{2}\omega^2q^2$ the energy $E=\tfrac{1}{2}p^2+\tfrac{1}{2}\omega^2q^2$ is multiplied by the same factor in every step (Eq. (38)),

$$E_{n+1}=\left(1+\omega^2\delta{t}^2\right)E_n \tag{38}$$

so that it grows without bound instead of oscillating in a band.

**Time-reversibility.** Let $S_1^{\mathrm{FE}}(\delta{t})$ denote one Euler step. Applying a step of size $-\delta{t}$ to the result of a step of size $\delta{t}$ does not return the starting state, but yields (Eq. (39))

$$\hat{r}\;\mapsto\;\hat{r}-\delta{t}^2\,\hat{a}(\hat{r}),\qquad \hat{v}\;\mapsto\;\hat{v}+\delta{t}\left[\hat{a}(\hat{r})-\hat{a}(\hat{r}+\delta{t}\,\hat{v})\right] \tag{39}$$

so that $S_1^{\mathrm{FE}}(-\delta{t})\circ S_1^{\mathrm{FE}}(\delta{t})\neq\mathrm{id}$.

### Conserved Quantities
**Linear momentum.** Summing Eq. (35) over all bodies with the masses as weights gives (Eq. (40))

$$\hat{P}_{n+1}=\hat{P}_n+\delta{t}\sum_{i=1}^{N}m_i\,\hat{a}_{i,n}=\hat{P}_n \tag{40}$$

because the pairwise forces in Eq. (4) are equal and opposite (Newton's third law), so that $\sum_i m_i\hat{a}_i=0$. Forward Euler therefore conserves $\hat{P}$ of Eq. (14) to rounding error, as long as the force routine respects this symmetry; the Barnes-Hut approximation does not (see the Barnes-Hut explanation).

**Angular momentum.** Inserting Eq. (34) and Eq. (35) into Eq. (16) and using $\hat{v}\times\hat{v}=0$ gives (Eq. (41))

$$\hat{L}_{n+1}=\hat{L}_n+\delta{t}\sum_{i=1}^{N}m_i\,\hat{r}_{i,n}\times\hat{a}_{i,n}+\delta{t}^2\sum_{i=1}^{N}m_i\,\hat{v}_{i,n}\times\hat{a}_{i,n}=\hat{L}_n+\delta{t}^2\sum_{i=1}^{N}m_i\,\hat{v}_{i,n}\times\hat{a}_{i,n} \tag{41}$$

The first sum vanishes, since the internal forces are central and cancel in pairs, but the second does not in general. For a body on a circular orbit, $\hat{v}\times\hat{a}$ is parallel to $\hat{L}$, so the error has the same sign in every step and accumulates. The error per step is $\mathcal{O}(\delta{t}^2)$, and the global error is again $\mathcal{O}(\delta{t})$.

**Energy.** The squared distance from the origin after one step is (Eq. (42))

$$\lvert\hat{r}_{n+1}\rvert^2=\lvert\hat{r}_n\rvert^2+2\,\delta{t}\,\hat{r}_n\cdot\hat{v}_n+\delta{t}^2\lvert\hat{v}_n\rvert^2 \tag{42}$$

For a circular orbit, or at an apsis of an elliptical one, $\hat{r}_n\perp\hat{v}_n$, so the middle term vanishes and the last one is positive: the body leaves the orbit along the tangent and ends up farther out. For a circular orbit with the angular frequency $\omega$, where $\lvert\hat{v}_n\rvert=\omega\lvert\hat{r}_n\rvert$, the radius grows by exactly $\sqrt{1+\omega^2\delta{t}^2}$ in the first step. The velocity update does the same to the speed, since $\lvert\hat{v}_{n+1}\rvert^2=\lvert\hat{v}_n\rvert^2+\delta{t}^2\lvert\hat{a}_n\rvert^2$ whenever $\hat{v}_n\cdot\hat{a}_n=0$. Both the potential and the kinetic energy of Eq. (12) rise, so the orbit spirals outward and the energy error grows systematically instead of oscillating in a bounded band, which is the behaviour described for non-symplectic methods in the section on the Hamiltonian (Hairer, Lubich & Wanner, 2006).

### Relation to the Symplectic Euler Method and the Leapfrog
The Euler step cannot be written as a composition of the kick and drift maps of Eq. (27), because both increments are evaluated at the same initial state. Composing the two maps instead, with the kick applied first, gives the symplectic Euler method (Eq. (43)),

$$S_1^{\mathrm{SE}}(\delta{t})=D(\delta{t})\circ K(\delta{t}),\qquad \hat{v}_{n+1}=\hat{v}_n+\hat{a}_n\,\delta{t},\quad \hat{r}_{n+1}=\hat{r}_n+\hat{v}_{n+1}\,\delta{t} \tag{43}$$

which is identical to `run_fe_1` with the two update loops swapped. It has the same cost and the same first order, but as a composition of exact kicks and drifts it is symplectic. The leapfrog step of Eq. (27) is, in turn, the composition of a symplectic Euler step of size $\delta{t}/2$ with its adjoint (Hairer, Lubich & Wanner, 2006). Forward Euler is retained here nonetheless, because it is the standard non-symplectic reference against which the symplectic integrators are compared.

### One Integration Step in the Code
Before the first step, `run_fe_1` evaluates the initial accelerations $\hat{a}_0$ of all bodies (Eq. (4)). Each timestep then consists of:
1. Move all bodies by a full timestep with $\hat{v}_n$ (Eq. (34)).
2. Update all velocities by a full timestep with $\hat{a}_n$ (Eq. (35)).
3. Re-evaluate the accelerations $\hat{a}_{n+1}$ of all bodies at the new displacements (Eq. (4)).

The accelerations computed in the third stage are reused by the first velocity update of the following step, so that one force evaluation per timestep is required, and the stored accelerations always belong to the stored displacements, as in the other integrators. All bodies are moved before any acceleration is recomputed, so that every force is evaluated for the same instant.



# Runge-Kutta 2nd Order (RK2)

## About

The Runge-Kutta methods (Runge, 1895; Kutta, 1901) raise the order of accuracy of Forward Euler by sampling the slope of the solution at several points inside a single timestep and advancing the system with a weighted average of these slopes. RK2 uses two such samples and solves the equations of motion to the second order of accuracy at the cost of two force evaluations per timestep. The variant implemented here is the explicit midpoint method: a trial half-step is taken along the tangent, and the slope found at the midpoint of that trial step is used for the full step. As with Forward Euler, the method is neither symplectic nor time-reversible, and it is therefore included as the second-order member of the non-symplectic comparison. The RK4 method presented afterwards is its fourth-order counterpart.

In [ ]:
"""
RK2 Implementation (explicit midpoint method; Runge, 1895).
Includes the stage evaluations, the displacement and velocity updates and the runtime computation.

Unlike the symplectic integrators above, the acceleration has to be evaluated at an intermediate
(trial) configuration of the whole system, which is not the state the bodies are currently in.
This is done by acc_stage, which is reused by the RK4 implementation below.
The detailed description of the algorithm will be provided later down below.
"""

# Evaluate the accelerations of ALL bodies at trial displacements
# bod_obj – a list of stellar bodies represented as objects.
# r_trial – list of trial displacement vectors, one for every body (same order as bod_obj)
# G – the constant of gravitational attraction
# e – softening factor to prevent division by 0 errors in case
#     the bodies get too close.
# acc_func – force routine, see run_fe_1
# NB: the displacements of the bodies are only changed temporarily. They are restored before
#     the function returns, so that the stored state of the simulation is never corrupted.
def acc_stage(bod_obj, r_trial, G, e, acc_func=acc):
    r_saved = [obj.r for obj in bod_obj]    # remember the true displacements

    try:
        for obj, r in zip(bod_obj, r_trial):
            obj.r = r
        return acc_func(G, bod_obj, e)      # forces at the trial configuration
    finally:
        for obj, r in zip(bod_obj, r_saved):
            obj.r = r                       # always restore the true displacements

# Trial displacement of an individual body: r_n + h*k_r
# obj – individual object with accessible properties
# h – (fraction of the) timestep that the stage advances by
# k_r – displacement slope of the previous stage
def stage_displ_rk(obj, h, k_r):
    return obj.r + h*k_r

# Trial velocity of an individual body: v_n + h*k_v
# obj – individual object with accessible properties
# h – (fraction of the) timestep that the stage advances by
# k_v – velocity slope (acceleration) of the previous stage
def stage_vel_rk(obj, h, k_v):
    return obj.v + h*k_v

# Compute Displacement of an individual body
# obj – individual object with accessible properties
# t – current timestep
# k2_r – displacement slope at the midpoint of the step
def displ_rk_2(obj, t, k2_r):
    return obj.r + t*k2_r

# Compute Velocity of an individual body
# obj – individual object with accessible properties
# t – current timestep
# k2_v – velocity slope (acceleration) at the midpoint of the step
def vel_rk_2(obj, t, k2_v):
    return obj.v + t*k2_v

# Simulation Runtime
# bod_obj – a list of stellar bodies represented as objects.
#           Contains objects themselves. Allows to access
#           individual parameters such as: mass, initial velocity,
#           and initial displacement.
# G – the constant of gravitational attraction
# e – softening factor to prevent division by 0 errors in case
#     the bodies get too close.
# dt – timestep of the simulation
# n_steps – number of timesteps to take
# acc_func – force routine used for every acceleration evaluation, see run_fe_1
# NB: the initial conditions (r, v, m) must be set on every body before calling this.
def run_rk_2(bod_obj, G, e, dt, n_steps, acc_func=acc):

    # Find initial accelerations of all bodies (the slope k1_v of the first step)
    for obj, a in zip(bod_obj, acc_func(G, bod_obj, e)):
        obj.a = a

    # Start the clock of the simulated Universe
    for _ in range(n_steps):

        # Stage 1 is free: k1_r = v_n and k1_v = a_n (already stored on every body).
        # Stage 2: take half a step with the stage 1 slopes and evaluate the slopes there (midpoint)
        r_mid = [stage_displ_rk(obj, 0.5*dt, obj.v) for obj in bod_obj]
        k2_r = [stage_vel_rk(obj, 0.5*dt, obj.a) for obj in bod_obj]
        k2_v = acc_stage(bod_obj, r_mid, G, e, acc_func)

        # Advance ALL bodies with the midpoint slopes. Everything is computed before anything is
        # assigned, so that no body uses the already updated state of another one
        r_new = [displ_rk_2(obj, dt, k) for obj, k in zip(bod_obj, k2_r)]
        v_new = [vel_rk_2(obj, dt, k) for obj, k in zip(bod_obj, k2_v)]

        for obj, r, v in zip(bod_obj, r_new, v_new):
            obj.r = r
            obj.v = v

        # Re-evaluate the forces once for the whole system at the new displacements (a_{n+1}).
        # They are the k1_v of the next step, so that no force is evaluated twice
        for obj, a in zip(bod_obj, acc_func(G, bod_obj, e)):
            obj.a = a

## Runge-Kutta 2nd Order Explanation
The explicit midpoint method, which is the form of RK2 implemented here, advances the system with the slopes of two stages: the slope at the beginning of the timestep, which is used only to reach a trial midpoint, and the slope at that midpoint, which is then used for the whole step. Because the second slope is a better estimate of the average slope over the step than the first one, the method gains one order of accuracy over Forward Euler. The price is that the force has to be evaluated at a configuration that the bodies never occupy.

**Notation.** As in the earlier explanations, the subscript $n$ denotes the timestep, so that $\hat{r}_n$, $\hat{v}_n$ and $\hat{a}_n=\hat{a}(\hat{r}_n)$ are the displacement, velocity and acceleration of a body at the time $t_n=n\,\delta{t}$, with $\hat{a}$ given by Eq. (4). The body index $i$ is omitted for readability and restored only where a sum over the bodies is required. The slopes of the stage $s$ are written $\hat{k}_{s,r}$ for the displacement and $\hat{k}_{s,v}$ for the velocity, in accordance with the variables `k1_r`, `k1_v`, `k2_r` and `k2_v` of the code.

### Update Equations
The equations of motion form the first-order system (Eq. (44))

$$\dot{\hat{r}}=\hat{v},\qquad\dot{\hat{v}}=\hat{a}(\hat{r}) \tag{44}$$

and every stage supplies one value of the right-hand side of Eq. (44). The first stage is the slope at the beginning of the step (Eq. (45)),

$$\hat{k}_{1,r}=\hat{v}_n,\qquad\hat{k}_{1,v}=\hat{a}_n \tag{45}$$

which is identical to the slope used by Forward Euler and is available without any further work. The second stage evaluates the slope at the midpoint $t_n+\tfrac{1}{2}\delta{t}$ of the trial half-step along the first slope (Eq. (46)),

$$\hat{k}_{2,r}=\hat{v}_n+\tfrac{1}{2}\hat{k}_{1,v}\,\delta{t},\qquad\hat{k}_{2,v}=\hat{a}\!\left(\hat{r}_n+\tfrac{1}{2}\hat{k}_{1,r}\,\delta{t}\right) \tag{46}$$

where the displacement slope is the velocity at the midpoint and the velocity slope is the gravitational acceleration at the trial displacement. The second stage alone is then used to advance the system:

**Displacement** (`displ_rk_2`, Eq. (47))

$$\hat{r}_{n+1}=\hat{r}_n+\hat{k}_{2,r}\,\delta{t} \tag{47}$$

**Velocity** (`vel_rk_2`, Eq. (48))

$$\hat{v}_{n+1}=\hat{v}_n+\hat{k}_{2,v}\,\delta{t} \tag{48}$$

The acceleration in Eq. (46) of one body depends on the displacements of all the other bodies through Eq. (4), so the trial displacements of the entire system have to exist before any $\hat{k}_{2,v}$ can be computed. This is the purpose of `acc_stage`: it temporarily places every body at its trial displacement, calls the force routine and restores the true displacements before returning, so that the stored state of the simulation is never corrupted. Since the trial configuration is passed to the force routine through the bodies themselves, any routine with the signature of `acc` can be used, including the Barnes-Hut routine `acc_bh`, which rebuilds its tree at every call.

Substituting Eq. (46) into Eq. (47) and Eq. (48) eliminates the stage variables (Eq. (49)),

$$\hat{r}_{n+1}=\hat{r}_n+\hat{v}_n\,\delta{t}+\tfrac{1}{2}\hat{a}_n\,\delta{t}^2,\qquad\hat{v}_{n+1}=\hat{v}_n+\hat{a}\!\left(\hat{r}_n+\tfrac{1}{2}\hat{v}_n\,\delta{t}\right)\delta{t} \tag{49}$$

so that the displacement update coincides with Eq. (5) of the Velocity Verlet method, while the velocity update uses the acceleration at the midpoint of the step instead of the average of the accelerations at its ends.

### Order of Accuracy
Let $\hat{y}=(\hat{r},\hat{v})$ and let $\hat{f}$ denote the right-hand side of Eq. (44), so that $\dot{\hat{y}}=\hat{f}(\hat{y})$. The Taylor series of the exact solution started from $\hat{y}_n$ is (Eq. (50))

$$\hat{y}(t_{n+1})=\hat{y}_n+\delta{t}\,\hat{f}(\hat{y}_n)+\tfrac{1}{2}\delta{t}^2\,\hat{f}'\hat{f}(\hat{y}_n)+\mathcal{O}(\delta{t}^3) \tag{50}$$

where $\hat{f}'$ is the Jacobian of $\hat{f}$. The slope of the second stage expands as (Eq. (51))

$$\hat{f}\!\left(\hat{y}_n+\tfrac{1}{2}\delta{t}\,\hat{f}(\hat{y}_n)\right)=\hat{f}(\hat{y}_n)+\tfrac{1}{2}\delta{t}\,\hat{f}'\hat{f}(\hat{y}_n)+\mathcal{O}(\delta{t}^2) \tag{51}$$

and multiplication by $\delta{t}$ reproduces Eq. (50) up to and including the term of order $\delta{t}^2$. The local error is therefore $\mathcal{O}(\delta{t}^3)$, and the global error, which accumulates over $\mathcal{O}(1/\delta{t})$ steps, is $\mathcal{O}(\delta{t}^2)$: halving the timestep reduces the error by a factor of four (Hairer, Nørsett & Wanner, 1993). The displacement can be examined explicitly. Since Eq. (49) is the Taylor polynomial of $\hat{r}(t)$ of degree two, the error of the displacement in one step is (Eq. (52))

$$\hat{r}(t_{n+1})-\hat{r}_{n+1}=\tfrac{1}{6}\dot{\hat{a}}_n\,\delta{t}^3+\mathcal{O}(\delta{t}^4) \tag{52}$$

where $\dot{\hat{a}}_n$ is the time derivative of the acceleration along the exact trajectory. The order of the method is the same as that of the leapfrog and the Velocity Verlet method, but it requires two force evaluations per step instead of one.

### Loss of the Geometric Properties
**Symplecticity.** A Runge-Kutta method with the coefficients $a_{ij}$ and the weights $b_i$ is symplectic for all Hamiltonian systems only if (Eq. (53))

$$b_i\,a_{ij}+b_j\,a_{ji}=b_i\,b_j\qquad\text{for all } i,j \tag{53}$$

(Hairer, Lubich & Wanner, 2006). For RK2 the only non-zero coefficient is $a_{21}=\tfrac{1}{2}$, with $b_1=0$ and $b_2=1$. An explicit method has $a_{ij}=0$ for $j\geq i$, so that Eq. (53) with $i=j$ reduces to $b_i^2=0$ for every stage, which contradicts $\sum_i b_i=1$. No explicit Runge-Kutta method can therefore be symplectic, and this holds for RK2 and RK4 alike. For the harmonic oscillator $\dot{q}=p$, $\dot{p}=-\omega^2q$, Eq. (46) to Eq. (48) give (Eq. (54))

$$q_{n+1}=\left(1-\tfrac{1}{2}\omega^2\delta{t}^2\right)q_n+\delta{t}\,p_n,\qquad p_{n+1}=\left(1-\tfrac{1}{2}\omega^2\delta{t}^2\right)p_n-\omega^2\delta{t}\,q_n \tag{54}$$

and the cross terms cancel in $p^2+\omega^2q^2$, so that the energy is multiplied by the same factor in every step (Eq. (55)),

$$E_{n+1}=\left(1+\tfrac{1}{4}\omega^4\delta{t}^4\right)E_n \tag{55}$$

The energy grows without bound, exactly as for Forward Euler in Eq. (38), but with the fourth power of $\omega\delta{t}$ instead of the second, which is why the growth is much slower at the same timestep.

**Time-reversibility.** An explicit Runge-Kutta method is never symmetric (Hairer, Lubich & Wanner, 2006): a step of size $-\delta{t}$ applied to the result of a step of size $\delta{t}$ does not return the starting state, as was the case for Forward Euler in Eq. (39).

### Conserved Quantities
**Linear momentum.** Summing Eq. (48) over all bodies with the masses as weights gives (Eq. (56))

$$\hat{P}_{n+1}=\hat{P}_n+\delta{t}\sum_{i=1}^{N}m_i\,\hat{k}_{2,v,i}=\hat{P}_n \tag{56}$$

because $\hat{k}_{2,v,i}$ is the output of the same pairwise force routine as Eq. (4), only evaluated at the trial displacements, and the forces therefore remain equal and opposite (Newton's third law), so that $\sum_i m_i\hat{k}_{2,v,i}=0$. The total momentum $\hat{P}$ of Eq. (14) is conserved to rounding error for any linear combination of such accelerations, provided the force routine respects this symmetry; the Barnes-Hut approximation does not.

**Angular momentum.** The total angular momentum $\hat{L}$ of Eq. (16) is a quadratic function of the state, and Eq. (53) is precisely the condition under which a Runge-Kutta method preserves quadratic invariants. Since it cannot be satisfied by an explicit method, $\hat{L}$ is conserved only to the order of the method: the error is bounded by the local error of $\mathcal{O}(\delta{t}^3)$ per step and by the global error of $\mathcal{O}(\delta{t}^2)$, and it accumulates instead of oscillating.

**Energy.** The Hamiltonian of Eq. (12) is not conserved exactly either. Its error is of the order of the method and, as in Eq. (55), it builds up steadily with the number of steps instead of staying inside a bounded band, which is the behaviour of non-symplectic methods described in the section on the Hamiltonian (Hairer, Lubich & Wanner, 2006). The sign of the drift depends on the orbit and cannot be read off Eq. (55), which is derived for the harmonic oscillator only.

### Relation to Other Methods
Eq. (49) shows that RK2 and the Velocity Verlet method update the displacements identically and differ only in the velocity update, where Velocity Verlet uses the average of $\hat{a}_n$ and $\hat{a}_{n+1}$. This small difference is sufficient to destroy the symplectic property. Replacing the midpoint slope in Eq. (46) by the average of the slopes at the two ends of the step gives Heun's method, a different second-order Runge-Kutta method of the same cost. The midpoint form is retained here because of its direct relation to Eq. (5), noted above.

### One Integration Step in the Code
Before the first step, `run_rk_2` evaluates the initial accelerations $\hat{a}_0$ of all bodies (Eq. (4)). Each timestep then consists of:
1. Take the first-stage slopes from the stored state, $\hat{k}_{1,r}=\hat{v}_n$ and $\hat{k}_{1,v}=\hat{a}_n$ (Eq. (45)).
2. Form the trial midpoint displacements $\hat{r}_n+\tfrac{1}{2}\hat{k}_{1,r}\delta{t}$ and the displacement slopes $\hat{k}_{2,r}$ of all bodies (Eq. (46)).
3. Evaluate the accelerations $\hat{k}_{2,v}$ of all bodies at the trial displacements with `acc_stage` (Eq. (46)).
4. Advance all displacements and all velocities with the second-stage slopes (Eq. (47) and Eq. (48)). Both new lists are computed before either is assigned, so that no body uses the already updated state of another one.
5. Re-evaluate the accelerations $\hat{a}_{n+1}$ of all bodies at the new displacements (Eq. (4)).

The accelerations computed in the last stage are the first-stage slopes $\hat{k}_{1,v}$ of the following step, so that exactly two force evaluations are required per timestep, and the stored accelerations always belong to the stored displacements, as in the other integrators.

# Runge-Kutta 4th Order (RK4)

## About

The classical fourth-order Runge-Kutta method (Kutta, 1901) extends the idea of RK2 to four stages: the slope is sampled once at the beginning of the timestep, twice at its midpoint and once at its end, and the four samples are combined with the weights $1:2:2:1$. It is the most widely used general-purpose integrator and, for a sufficiently small timestep, it is far more accurate over short simulations than any of the second-order methods of this notebook. The accuracy is bought with four force evaluations per timestep, which is the dominant cost of an N-body simulation, and it does not extend to long simulations, since RK4 is not symplectic and therefore shows a systematic energy drift.

In [ ]:
"""
RK4 Implementation (classical fourth-order Runge-Kutta method; Kutta, 1901).
Includes the displacement and velocity updates and the runtime computation.

The stage helper functions stage_displ_rk and stage_vel_rk and the trial force evaluation
acc_stage are carried over from the RK2 cell above.
The detailed description of the algorithm will be provided later down below.
"""

# Compute Displacement of an individual body
# obj – individual object with accessible properties
# t – current timestep
# k1_r, k2_r, k3_r, k4_r – displacement slopes of the four stages
def displ_rk_4(obj, t, k1_r, k2_r, k3_r, k4_r):
    return obj.r + (t/6.0)*(k1_r + 2.0*k2_r + 2.0*k3_r + k4_r)

# Compute Velocity of an individual body
# obj – individual object with accessible properties
# t – current timestep
# k1_v, k2_v, k3_v, k4_v – velocity slopes (accelerations) of the four stages
def vel_rk_4(obj, t, k1_v, k2_v, k3_v, k4_v):
    return obj.v + (t/6.0)*(k1_v + 2.0*k2_v + 2.0*k3_v + k4_v)

# Simulation Runtime
# bod_obj – a list of stellar bodies represented as objects.
#           Contains objects themselves. Allows to access
#           individual parameters such as: mass, initial velocity,
#           and initial displacement.
# G – the constant of gravitational attraction
# e – softening factor to prevent division by 0 errors in case
#     the bodies get too close.
# dt – timestep of the simulation
# n_steps – number of timesteps to take
# acc_func – force routine used for every acceleration evaluation, see run_fe_1
# NB: the initial conditions (r, v, m) must be set on every body before calling this.
def run_rk_4(bod_obj, G, e, dt, n_steps, acc_func=acc):

    # Find initial accelerations of all bodies (the slope k1_v of the first step)
    for obj, a in zip(bod_obj, acc_func(G, bod_obj, e)):
        obj.a = a

    # Start the clock of the simulated Universe
    for _ in range(n_steps):

        # Stage 1 is free: the slopes at the start of the step are k1_r = v_n and k1_v = a_n
        k1_r = [obj.v for obj in bod_obj]
        k1_v = [obj.a for obj in bod_obj]

        # Stage 2: half a step with the stage 1 slopes
        r_2 = [stage_displ_rk(obj, 0.5*dt, k) for obj, k in zip(bod_obj, k1_r)]
        k2_r = [stage_vel_rk(obj, 0.5*dt, k) for obj, k in zip(bod_obj, k1_v)]
        k2_v = acc_stage(bod_obj, r_2, G, e, acc_func)

        # Stage 3: half a step again, now with the stage 2 slopes
        r_3 = [stage_displ_rk(obj, 0.5*dt, k) for obj, k in zip(bod_obj, k2_r)]
        k3_r = [stage_vel_rk(obj, 0.5*dt, k) for obj, k in zip(bod_obj, k2_v)]
        k3_v = acc_stage(bod_obj, r_3, G, e, acc_func)

        # Stage 4: a FULL step with the stage 3 slopes
        r_4 = [stage_displ_rk(obj, dt, k) for obj, k in zip(bod_obj, k3_r)]
        k4_r = [stage_vel_rk(obj, dt, k) for obj, k in zip(bod_obj, k3_v)]
        k4_v = acc_stage(bod_obj, r_4, G, e, acc_func)

        # Advance ALL bodies with the weighted average of the four slopes (weights 1, 2, 2, 1).
        # Everything is computed before anything is assigned, so that no body uses the already
        # updated state of another one
        r_new = [displ_rk_4(obj, dt, k1_r[i], k2_r[i], k3_r[i], k4_r[i]) for i, obj in enumerate(bod_obj)]
        v_new = [vel_rk_4(obj, dt, k1_v[i], k2_v[i], k3_v[i], k4_v[i]) for i, obj in enumerate(bod_obj)]

        for obj, r, v in zip(bod_obj, r_new, v_new):
            obj.r = r
            obj.v = v

        # Re-evaluate the forces once for the whole system at the new displacements (a_{n+1}).
        # They are the k1_v of the next step, so that every step costs exactly four force evaluations
        for obj, a in zip(bod_obj, acc_func(G, bod_obj, e)):
            obj.a = a

## Runge-Kutta 4th Order Explanation
The classical RK4 method advances the system with a weighted average of four slopes. Each of the three trial stages is built from the slope of the stage before it, so that the later slopes correct the errors of the earlier ones, and the weights are chosen so that the error terms of the average cancel up to the fourth power of the timestep. The cost is four force evaluations per step, each of them at a trial configuration of the whole system, which is carried out by the routine `acc_stage` introduced for RK2.

**Notation.** The notation of the RK2 explanation is retained: the subscript $n$ denotes the timestep, $\hat{a}_n=\hat{a}(\hat{r}_n)$ is given by Eq. (4), the body index $i$ is omitted, and $\hat{k}_{s,r}$ and $\hat{k}_{s,v}$ are the displacement and velocity slopes of the stage $s=1,\dots,4$. The right-hand side $\hat{f}$ of the first-order system is that of Eq. (44).

### Update Equations
The four stages are evaluated in turn. The first one is the slope at the beginning of the step and requires no work (Eq. (57)),

$$\hat{k}_{1,r}=\hat{v}_n,\qquad\hat{k}_{1,v}=\hat{a}_n \tag{57}$$

The second stage is the slope at the midpoint of a trial half-step along the first slope, as in Eq. (46) (Eq. (58)),

$$\hat{k}_{2,r}=\hat{v}_n+\tfrac{1}{2}\hat{k}_{1,v}\,\delta{t},\qquad\hat{k}_{2,v}=\hat{a}\!\left(\hat{r}_n+\tfrac{1}{2}\hat{k}_{1,r}\,\delta{t}\right) \tag{58}$$

The third stage returns to the midpoint, but now follows the improved second slope (Eq. (59)),

$$\hat{k}_{3,r}=\hat{v}_n+\tfrac{1}{2}\hat{k}_{2,v}\,\delta{t},\qquad\hat{k}_{3,v}=\hat{a}\!\left(\hat{r}_n+\tfrac{1}{2}\hat{k}_{2,r}\,\delta{t}\right) \tag{59}$$

and the fourth stage takes a trial step over the whole timestep along the third slope, in order to sample the slope at its end (Eq. (60)),

$$\hat{k}_{4,r}=\hat{v}_n+\hat{k}_{3,v}\,\delta{t},\qquad\hat{k}_{4,v}=\hat{a}\!\left(\hat{r}_n+\hat{k}_{3,r}\,\delta{t}\right) \tag{60}$$

The system is then advanced with the weighted average of the four slopes:

**Displacement** (`displ_rk_4`, Eq. (61))

$$\hat{r}_{n+1}=\hat{r}_n+\frac{\delta{t}}{6}\left(\hat{k}_{1,r}+2\hat{k}_{2,r}+2\hat{k}_{3,r}+\hat{k}_{4,r}\right) \tag{61}$$

**Velocity** (`vel_rk_4`, Eq. (62))

$$\hat{v}_{n+1}=\hat{v}_n+\frac{\delta{t}}{6}\left(\hat{k}_{1,v}+2\hat{k}_{2,v}+2\hat{k}_{3,v}+\hat{k}_{4,v}\right) \tag{62}$$

Since the displacement slopes in Eq. (58) to Eq. (60) are built from the velocity slopes, the stage variables of the displacement can be eliminated (Eq. (63)),

$$\hat{r}_{n+1}=\hat{r}_n+\hat{v}_n\,\delta{t}+\frac{\delta{t}^2}{6}\left(\hat{k}_{1,v}+\hat{k}_{2,v}+\hat{k}_{3,v}\right) \tag{63}$$

so that the displacement is that of a body in uniformly accelerated motion with the average of the first three accelerations, and the fourth acceleration $\hat{k}_{4,v}$ enters only the velocity. The weights $1:2:2:1$ are those of Simpson's rule: if the right-hand side of Eq. (44) depended on the time only, Eq. (61) and Eq. (62) would reduce to the quadrature (Eq. (64))

$$y_{n+1}=y_n+\frac{\delta{t}}{6}\left[f(t_n)+4f\!\left(t_n+\tfrac{1}{2}\delta{t}\right)+f(t_{n+1})\right] \tag{64}$$

in which the two midpoint slopes together carry the weight four (Hairer, Nørsett & Wanner, 1993).

### Order of Accuracy
Expanding the exact solution and the four stages in Taylor series and comparing the coefficients shows that the weights and the stage coefficients of Eq. (57) to Eq. (62) satisfy all conditions for the terms up to and including $\delta{t}^4$, but not for the term of order $\delta{t}^5$ (Hairer, Nørsett & Wanner, 1993). The local error is therefore $\mathcal{O}(\delta{t}^5)$ and the global error is $\mathcal{O}(\delta{t}^4)$: halving the timestep reduces the error by a factor of sixteen. This is the highest order of the non-symplectic methods of this notebook, and the same order as that of the Velocity Verlet 4th order and the Forest-Ruth integrators. In terms of the cost, one RK4 step is as expensive as four leapfrog steps of the size $\delta{t}/4$, which is the appropriate basis for a comparison at the equal number of force evaluations.

### Loss of the Geometric Properties
**Symplecticity and time-reversibility.** RK4 is an explicit Runge-Kutta method, so that the argument following Eq. (53) applies without change: it cannot be symplectic and cannot be symmetric (Hairer, Lubich & Wanner, 2006). For a linear problem $\dot{y}=\lambda y$ one step of RK4 multiplies the solution by (Eq. (65))

$$R(z)=1+z+\frac{z^2}{2}+\frac{z^3}{6}+\frac{z^4}{24},\qquad z=\lambda\,\delta{t} \tag{65}$$

which is the Taylor polynomial of $e^z$ of degree four (Hairer, Nørsett & Wanner, 1993). The harmonic oscillator of Eq. (54) has the eigenvalues $\lambda=\pm i\omega$ and its energy is proportional to the squared modulus of the corresponding amplitude, so that the energy is multiplied in every step by (Eq. (66))

$$\left\lvert R(i\omega\delta{t})\right\rvert^2=1-\frac{(\omega\delta{t})^6}{72}+\frac{(\omega\delta{t})^8}{576} \tag{66}$$

For $\omega\delta{t}<2\sqrt{2}$ this factor is smaller than one, and the energy decays by the relative amount $(\omega\delta{t})^6/72$ per step. This is a much smaller loss than the gain of Eq. (55) at the same timestep, but it is still systematic: the energy error is not an oscillation inside a bounded band as for the symplectic methods, but a drift that accumulates with the number of steps. For $\omega\delta{t}>2\sqrt{2}$ the factor exceeds one and the numerical solution grows without bound, which sets the largest usable timestep for a system with the angular frequency $\omega$.

### Conserved Quantities
**Linear momentum.** The velocity update of Eq. (62) is a linear combination of four accelerations, each of which is the output of the pairwise force routine of Eq. (4) at some configuration and therefore satisfies $\sum_i m_i\hat{k}_{s,v,i}=0$, so that (Eq. (67))

$$\hat{P}_{n+1}=\hat{P}_n+\frac{\delta{t}}{6}\sum_{i=1}^{N}m_i\left(\hat{k}_{1,v,i}+2\hat{k}_{2,v,i}+2\hat{k}_{3,v,i}+\hat{k}_{4,v,i}\right)=\hat{P}_n \tag{67}$$

so that $\hat{P}$ of Eq. (14) is conserved to rounding error whenever the force routine respects Newton's third law, and not otherwise, as for the Barnes-Hut routine.

**Angular momentum and energy.** As for RK2, $\hat{L}$ of Eq. (16) is a quadratic invariant and is preserved only by methods that satisfy Eq. (53), and the Hamiltonian of Eq. (12) is not preserved by any Runge-Kutta method in general. Both errors are bounded by the global error of $\mathcal{O}(\delta{t}^4)$ and are therefore very small for a small timestep, but they have no mechanism to remain bounded in time. For short simulations RK4 can therefore be more accurate than the symplectic second-order methods, which is the "short-term success" of the non-symplectic methods referred to in the introduction of this section, whereas for a large number of steps the drift accumulates and eventually dominates.

### One Integration Step in the Code
Before the first step, `run_rk_4` evaluates the initial accelerations $\hat{a}_0$ of all bodies (Eq. (4)). Each timestep then consists of:
1. Take the first-stage slopes from the stored state, $\hat{k}_{1,r}=\hat{v}_n$ and $\hat{k}_{1,v}=\hat{a}_n$ (Eq. (57)).
2. Form the trial displacements $\hat{r}_n+\tfrac{1}{2}\hat{k}_{1,r}\delta{t}$ of all bodies, compute $\hat{k}_{2,r}$ and evaluate $\hat{k}_{2,v}$ with `acc_stage` (Eq. (58)).
3. Repeat with the second-stage slopes to obtain $\hat{k}_{3,r}$ and $\hat{k}_{3,v}$ (Eq. (59)).
4. Repeat with the third-stage slopes and the full timestep to obtain $\hat{k}_{4,r}$ and $\hat{k}_{4,v}$ (Eq. (60)).
5. Advance all displacements and all velocities with the weights $1:2:2:1$ (Eq. (61) and Eq. (62)). Both new lists are computed before either is assigned, so that no body uses the already updated state of another one.
6. Re-evaluate the accelerations $\hat{a}_{n+1}$ of all bodies at the new displacements (Eq. (4)).

The accelerations computed in the last stage are the first-stage slopes $\hat{k}_{1,v}$ of the following step, so that exactly four force evaluations are required per timestep: three at the trial configurations and one at the new state. The trial displacements are never stored on the bodies, so that the state of the simulation is always the state at the end of the last completed step.

## References
- Aarseth, S. J., Hénon, M. and Wielen, R. (1974) A comparison of numerical methods for the study of star cluster dynamics. *Astronomy and Astrophysics*, 37, 183–187.
- Barnes, J. and Hut, P. (1986) A hierarchical $\mathcal{O}(N\log N)$ force-calculation algorithm. *Nature*, 324, 446–449.
- Chenciner, A. and Montgomery, R. (2000) A remarkable periodic solution of the three-body problem in the case of equal masses. *Annals of Mathematics*, 152(3), 881–901.
- Dehnen, W. and Read, J. I. (2011) N-body simulations of gravitational dynamics. *The European Physical Journal Plus*, 126, 55.
- Forest, E. and Ruth, R. D. (1990) Fourth-order symplectic integration. *Physica D: Nonlinear Phenomena*, 43(1), 105–117.
- Hairer, E., Lubich, C. and Wanner, G. (2006) *Geometric Numerical Integration: Structure-Preserving Algorithms for Ordinary Differential Equations*. 2nd edn. Berlin: Springer.
- Hairer, E., Nørsett, S. P. and Wanner, G. (1993) *Solving Ordinary Differential Equations I: Nonstiff Problems*. 2nd edn. Berlin: Springer.
- Kutta, W. (1901) Beitrag zur näherungsweisen Integration totaler Differentialgleichungen. *Zeitschrift für Mathematik und Physik*, 46, 435–453.
- Moore, C. (1993) Braids in classical dynamics. *Physical Review Letters*, 70(24), 3675–3679.
- Omelyan, I. P., Mryglod, I. M. and Folk, R. (2002) Optimized Forest–Ruth- and Suzuki-like algorithms for integration of motion in many-body systems. *Computer Physics Communications*, 146(2), 188–202.
- Plummer, H. C. (1911) On the problem of distribution in globular star clusters. *Monthly Notices of the Royal Astronomical Society*, 71, 460–470.
- Runge, C. (1895) Über die numerische Auflösung von Differentialgleichungen. *Mathematische Annalen*, 46, 167–178.
- Salmon, J. K. and Warren, M. S. (1994) Skeletons from the treecode closet. *Journal of Computational Physics*, 111(1), 136–155.
- Simó, C. (2002) Dynamical properties of the figure eight solution of the three-body problem. In: Chenciner, A., Cushman, R., Robinson, C. and Xia, Z. J. (eds) *Celestial Mechanics: Dedicated to Donald Saari for his 60th Birthday*. Contemporary Mathematics, 292. Providence: American Mathematical Society, 209–228.
- Springel, V. (2005) The cosmological simulation code GADGET-2. *Monthly Notices of the Royal Astronomical Society*, 364(4), 1105–1134.
- Swope, W. C., Andersen, H. C., Berens, P. H. and Wilcox, K. R. (1982) A computer simulation method for the calculation of equilibrium constants for the formation of physical clusters of molecules: application to small water clusters. *The Journal of Chemical Physics*, 76(1), 637–649.
- Verlet, L. (1967) Computer "experiments" on classical fluids. I. Thermodynamical properties of Lennard-Jones molecules. *Physical Review*, 159(1), 98–103.
- Yoshida, H. (1990) Construction of higher order symplectic integrators. *Physics Letters A*, 150(5–7), 262–268.